# GeoNUSAF Part 2 / Stage 3 — Chained sampling: fake mask -> fake image -> fake pair

**Kathmandu Valley synthetic image-mask pair factory — 6 classes + ignore, 512x512**

Structural twin of `segformer-b0-new.ipynb`, `geonusaf-stage1-maskddpm-onehot.ipynb` and
`geonusaf-stage2-mask2image.ipynb`: same 25 sections, same config-dataclass pattern, same asserts,
same persistence.

---

## What this notebook does

Stage 1 learned the *layout* distribution. Stage 2 learned the *appearance* conditional. Neither
alone produces a training example. This notebook chains them:

```
  z ~ N(0,I)  --[Stage 1 UNet2DModel, DDIM]-->  fake mask (128, codes 0..6)
                --[smooth_mask + greedy_match]-->  fake mask (512, values 0..5 / 255)
                --[Stage 2 SD1.5 + LoRA + control branch, DDIM]-->  fake image (512, RGB)
                --> (genimage{i}.png, genmask{i}.png)
```

and produces **three independent draws** at ratios R = 1x, 2x, 3x of the fold-1 real training set:

| set | pairs | purpose |
|---|---|---|
| R1 | 804 | 1x the 804 real fold-1 training pairs |
| R2 | 1608 | 2x |
| R3 | 2412 | 3x |

**Independent, not nested.** R1 is not a prefix of R3. Each set gets its own mask pool at its own
seed and its own image noise, so the three points on the Stage-4 augmentation curve are three
independent samples from the generator rather than three windows onto one sample. This costs 3x the
sampling but it is the only version where the R-sweep can be read as a curve. Say so in the paper.

## Fold 1, not fold 0

Both trained generators are fold-1 models (`...-block-fold1`). Fold 1 is the only block fold whose
validation block contains enough River to measure it, and the part-1 fold-1 scores leave real
headroom for the layout-fidelity ceiling. The frozen segmenter used for scoring is therefore also
the **fold-1** part-1 checkpoint. Mixing a fold-0 segmenter into a fold-1 pipeline would be a
silent leak, and section 8 asserts the split SHA1 agrees with both upstream stages.

## The one number that decides whether Part 2 is publishable

Section 13's **memorization check**. If the fakes are near-copies of the training tiles, then every
downstream mIoU gain in Stage 4 is a memorisation artefact and the whole result is void. It is run
at three levels — Inception feature space, raw pixels, and mask space — each against the same
statistic computed on *held-out real* tiles as the baseline. A ratio near or above 1.0 means the
generator is not reciting. **This is reported in the paper whichever way it comes out.**

## What is deliberately NOT here

No training. Nothing in this notebook takes a gradient step. Both generators are loaded frozen from
Hugging Face, and the sampling hyper-parameters are **inherited from the checkpoints' own embedded
`cfg`** rather than retyped, so a Stage-2 run that finished at a different guidance scale than the
one you remember cannot silently disagree with this notebook.

## Run budget

~4800 images at DDIM 30 on a T4 is roughly 7 hours of pure sampling, so this is a **two-session
job**. Section 17 writes every pair to disk the moment it is produced and section 18 resumes by
scanning what is already there, so a session that dies at pair 3100 restarts at pair 3100. The mask
pools are cached to `.npy` and pushed to the Hub, so a resumed session paints the *same* masks.

## 1. Environment

Same install line as Stage 2, because this notebook instantiates the same objects: `diffusers` for
both UNets and the schedulers, `peft` for the LoRA adapter, `transformers` for CLIP and for the
frozen SegFormer. `bitsandbytes` is not needed — there is no optimizer here — but the import is
cheap and keeps the environment identical to the run that produced the weights.

The `torchao` shim is carried over verbatim from Stage 2. `peft`'s LoRA dispatcher calls
`is_torchao_available()`, which **raises** on Kaggle's torchao 0.10 instead of returning `False`.
Nothing here uses torchao. Without the shim `add_adapter` dies and the Stage-2 weights cannot be
loaded at all.

Versions are printed rather than pinned beyond a floor. Record what prints here in the paper's
reproducibility section.

In [ ]:
!pip install -q "diffusers>=0.27.0" "peft>=0.10.0" ptflops==0.7.3 2>&1 | tail -2

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # set BEFORE cuda initialises

import re, gc, json, time, math, random, warnings, itertools, copy, hashlib, tarfile, shutil, csv
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import diffusers, transformers, peft

# ---- torchao shim: carried over from Stage 2, see the note above -------------------------
import importlib
def _disable_torchao_in_peft():
    hit = []
    for name in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.tuners.lora.model",
                 "peft.tuners.tuners_utils", "peft.utils.other"):
        try: mod = importlib.import_module(name)
        except Exception: continue
        if getattr(mod, "is_torchao_available", None) is not None:
            mod.is_torchao_available = lambda: False
            hit.append(name.split(".")[-1])
    return hit

try:
    import torchao
    _tao = getattr(torchao, "__version__", "?")
except Exception:
    _tao = None
_shim = _disable_torchao_in_peft()
print(f"torchao {_tao} | peft torchao-dispatcher shim applied to: {_shim or 'nothing (not needed)'}")

print("torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| transformers", transformers.__version__, "| peft", peft.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print(f"free disk on /kaggle/working: "
      f"{shutil.disk_usage('/kaggle/working').free/1e9:.1f} GB  "
      f"(~2.4 GB needed for 4824 pairs + tarballs)")

## 2. Configuration

**Stage 3 runs on fold 1 only**, because both generators are fold-1 models. That is not a
convenience, it is the leakage rule as it now stands: the generators saw fold-1 TRAIN tiles and
nothing else, the frozen scorer is the fold-1 part-1 checkpoint, and the Stage-4 augmentation
experiment will be reported on fold 1. Section 8 asserts all three agree by SHA1 of the train id
list.

### Sampling hyper-parameters are inherited, not retyped

`DDIM_STEPS`, `GUIDANCE`, `COND_SCALE`, `CANNY_SCALE` and `USE_CANNY` all default to `None`, which
means **take the value from the Stage-2 checkpoint's embedded `cfg`**. This exists because a long
run gets retuned mid-flight and the value you remember is not necessarily the value the weights
were selected under. Section 12 prints the resolved values and section 20 records them. Set any of
them to a number only if you deliberately want to sample off-config, and then say so in the paper.

### The R ladder

`R_MULTIPLIERS = (1, 2, 3)` against `n_real = len(tr_recs)` = 804. Three independent draws:
804 + 1608 + 2412 = 4824 pairs. `MASK_POOL_RATIO` oversamples the mask pool before the
distribution-matched filter, exactly as Stage 1's `N_POOL` / `N_GENERATE` did, so the filter has
something to select against instead of being a no-op.

### Two frozen scorers, not one

`SEG_REPO_BLOCK` is the fold-1 block model — the one that matches this pipeline's split and is the
headline. `SEG_REPO_RANDOM` is the fold-1 random-split model, used only for **inter-model
agreement**: if two independently trained segmenters both read the same layout out of a fake image,
that is far stronger evidence than one of them doing so. Neither is ever fine-tuned here.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (fixed for stage 3 -- see the leakage rule) ----
    SPLIT_MODE: str = "block"          # stage 3 is block-only; "random" would break the protocol
    FOLD: int = 1                      # both generators are fold-1 models

    # ---- the two trained generators, loaded frozen from the Hub ----
    S1_REPO: str = "sugam24/geonusaf-stage1-maskddpm-onehot-block-fold1"
    S2_REPO: str = "Pranilllllll/geonusaf-stage2-mask2image-block-fold1-D"

    # ---- frozen part-1 scorers (INFERENCE ONLY, never fine-tuned here) ----
    SEG_REPO_BLOCK:  str = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1"
    SEG_REPO_RANDOM: str = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-random-fold1"
    USE_SECOND_SCORER: bool = True     # inter-model agreement on the same fakes

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/satellite-datasets/satellite_dataset"   # auto-detected below
    CACHE_DIR: str = "/kaggle/working/cache512"
    # If you saved the 512 cache as a Kaggle dataset (the part-1 and Stage-2 notebooks build the
    # same one), point this at it and section 6 becomes a no-op.
    PREBUILT_CACHE: Optional[str] = None
    IMG_SIZE: int = 512                # interface resolution, both stages
    GEN_SIZE: int = 128                # Stage 1's diffusion resolution; mask stats live here
    LATENT_SIZE: int = 64              # IMG_SIZE / 8, the SD VAE downsample factor
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255            # on disk, and in the interface contract
    IGNORE_CODE: int = 6               # inside both models
    N_CODES: int = 7                   # 0..5 + ignore
    N_COND_CH: int = 7                 # one condition channel per code
    PRELOAD_RAM: bool = True

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary
    # Paste the sha1 printed by Stage 1 / Stage 2 section 8 to hard-assert three-way agreement.
    UPSTREAM_TRAIN_SHA1: Optional[str] = None

    # ---- the R ladder ----
    R_MULTIPLIERS: tuple = (1, 2, 3)   # x the real fold-1 training set (804 pairs)
    INDEPENDENT_DRAWS: bool = True     # True: three separate pools. False: nested prefixes (cheap)
    MASK_POOL_RATIO: float = 2.5       # pool size / kept size, before the distribution filter
    MASK_SEED_BASE: int = 20260829     # per-R mask seed = base + 100000*R
    IMAGE_SEED_BASE: int = 777         # per-R image seed = base + 1000*R

    # ---- stage 1 sampling (mask) ----
    S1_DDIM_STEPS: Optional[int] = None    # None -> FINAL_DDIM_STEPS from the S1 checkpoint (200)
    S1_BATCH: int = 64                     # 12 M params at 128 px: 64 is comfortable on a T4
    AMB_MARGIN: float = 0.25               # top1 - top2 below this counts as an ambiguous pixel
    # Post-processing knobs. None -> run the sweep in section 10 against the real fold-1 masks and
    # use whatever it picks, once, for all three R sets so they stay comparable.
    # Directory holding masks_R*.npy from an earlier run (e.g. the Output of a failed Kaggle
    # version, attached as a dataset). Checked BEFORE the Hub. Mask sampling is ~7.8 h; never
    # let a fixable downstream crash cost you those hours twice.
    MASK_RESTORE_DIR: Optional[str] = None
    # Directory holding already-painted pairs from an earlier session: either extracted
    # R1/ R2/ R3/ (or fake_pairs_R1/ ...) folders, or the tarballs themselves. Point this at a
    # failed/partial version's Output attached as a dataset. Section 17 copies in every pair
    # that is missing locally, so a resumed run repaints only what genuinely does not exist.
    PAIR_RESTORE_DIR: Optional[str] = None
    RESTORE_PAIRS_FROM_HF: bool = True     # also pull any pair tarballs already on the Hub
    PP_SWEEP: bool = True
    PP_SWEEP_N: int = 600
    PP_MODAL: Optional[int] = None
    PP_CLOSE: Optional[int] = None
    PP_THIN:  Optional[int] = None
    PP_BULK:  Optional[int] = None
    FILTER_MASKS: bool = True              # despeckle + distribution-matched selection

    # ---- stage 2 sampling (image) -- None means INHERIT from the S2 checkpoint ----
    DDIM_STEPS:  Optional[int]   = None
    GUIDANCE:    Optional[float] = None
    COND_SCALE:  Optional[float] = None
    CANNY_SCALE: Optional[float] = None
    USE_CANNY:   Optional[bool]  = None
    IMG_BATCH: int = 8                     # VRAM peak lives here, not in the mask sampler
    WRITE_EVERY: int = 25                  # pairs between progress prints / resume checkpoints
    # OFF BY DEFAULT, and it should stay off unless you have time to spare for an experiment.
    # The single-device path is the one that has actually run this stack end to end. The
    # multi-GPU path (one replica per device, worker threads on a shared queue) is a speed
    # optimisation that failed twice on Kaggle's T4 x2 with a CUDA illegal memory access that
    # survived both torch.cuda.set_device() in the worker and a torch.cuda.device() context
    # around the sampler. If you turn it on, section 14 benches every device first and DROPS
    # any that fails, falling back to whatever works -- it will never take the run down.
    MULTI_GPU: bool = False
    # Cast the control branch to fp16. Stage 2 SAVED control_state and lora_state as fp16, so
    # this discards no precision the checkpoint ever had, and roughly halves the branch's
    # memory bandwidth -- which is where a T4 actually spends its time.
    S2_FP16: bool = True
    PUSH_PARTIAL: bool = True              # tar+push incomplete sets if the session runs out
    # Push finished pairs to the Hub every N pairs, as small incremental chunks, WHILE the
    # run is still going. This is the only thing that makes a killed session cheap: without
    # it the most you can lose is the whole session's painting, with it the most you can lose
    # is N pairs. Set to 0 to disable (don't).
    CHECKPOINT_EVERY: int = 250

    # ---- evaluation ----
    EVAL_N: int = 256                  # fakes scored for layout fidelity, KID and FID (per R set)
    EVAL_BATCH: int = 8
    KID_SUBSET_SIZE: int = 50
    KID_SUBSETS: int = 100
    LAYOUT_RATIO_FLOOR: float = 0.60   # gen mIoU / real-image mIoU -- the honest reading
    COND_SOURCE_ABLATION: bool = True  # also paint REAL val masks, to isolate whether S1 masks are
                                       # out of distribution for the S2 control branch
    COND_ABL_N: int = 128
    NN_CHECK_N: int = 300              # fakes scored in the memorization check
    NN_TOPK_FIG: int = 8               # closest fake/real pairs drawn in section 23

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 2
    AMP: bool = True
    MAX_HOURS: float = 11.0            # stop before Kaggle's 12 h cap, then resume next session

    # ---- persistence ----
    HF_USER: str = "sugam24"
    HF_PREFIX: str = "geonusaf-stage3-fakepairs"
    PUSH_SHARDS: bool = True           # push one tarball per R set (~0.4 / 0.8 / 1.2 GB)
    WANDB_PROJECT: str = "geonusaf-baselines"
    OUT_DIR: str = "/kaggle/working/out"
    PAIR_DIR: str = "/kaggle/working/fake_pairs"     # PAIR_DIR/R1, /R2, /R3

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.PAIR_DIR, exist_ok=True)

assert cfg.SPLIT_MODE == "block" and cfg.FOLD == 1, \
    "stage 3 is defined on block fold 1 only -- both generators are fold-1 models"
assert cfg.N_COND_CH == cfg.N_CODES, "one condition channel per code"
assert cfg.IGNORE_CODE == cfg.N_CLASSES, "ignore must be the code just past the semantic classes"

CODE_NAMES = list(cfg.CLASS_NAMES) + ["Ignore"]        # 7 code slots, index == code
RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
def R_DIR(r): return f"{cfg.PAIR_DIR}/R{r}"

print("RUN:", RUN_NAME, "| HF repo:", HF_REPO, "| results tag:", tag)
print(f"chain: {cfg.S1_REPO}\n    -> {cfg.S2_REPO}")
print(f"scored by: {cfg.SEG_REPO_BLOCK}")
if cfg.USE_SECOND_SCORER: print(f"     and: {cfg.SEG_REPO_RANDOM}  (agreement only)")
print(f"R ladder {cfg.R_MULTIPLIERS} | "
      f"{'INDEPENDENT draws' if cfg.INDEPENDENT_DRAWS else 'NESTED prefixes'} | "
      f"mask pool ratio {cfg.MASK_POOL_RATIO}")
print("sampling hyper-parameters: inherited from the checkpoints unless overridden in CFG")

## 3. Determinism — deliberately OFF here

Same setting as Stages 1 and 2, for the same reason, and it is not an oversight.
`torch.use_deterministic_algorithms(True)` routes part of these UNets through sort-based kernels and
costs roughly **6x wall-clock** on this workload. At 4824 images that is the difference between two
sessions and twelve.

Reproducibility here does not rest on cuDNN determinism anyway. It rests on the seeds: every mask
batch and every image batch draws its noise from an explicitly seeded `torch.Generator` on CPU
(`MASK_SEED_BASE`, `IMAGE_SEED_BASE`, offset by the batch start index), so re-running any batch
reproduces the same latents on any machine. What can differ run to run is the last bit of the fp16
attention accumulation, which moves a handful of boundary pixels and nothing else.

`cudnn.benchmark = True` is correct here: every batch in this notebook has fixed shape.

One consequence of the multi-GPU sampler in section 17 worth stating up front: **the latent for
pair `i` is drawn from a generator seeded by `i` alone**, not by its batch, so pair 3100 is the
same image whether it was painted first or last, on GPU 0 or GPU 1, in one session or in four.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True          # fixed shapes -> autotune pays for itself
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

set_seed(cfg.SEED)
T_START = time.time()
def hours(): return (time.time() - T_START) / 3600.0
def budget_left(): return cfg.MAX_HOURS - hours()

print(f"seed {cfg.SEED} | deterministic algorithms OFF (see the note above) | "
      f"cudnn.benchmark {torch.backends.cudnn.benchmark}")
print(f"session budget {cfg.MAX_HOURS} h -- section 18 stops cleanly and resumes next session")

## 4. Filename parsing and base-tile grouping

Identical to the part-1 notebooks and to Stages 1 and 2, and copied rather than rewritten for
exactly that reason. Convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror
with the `mask` prefix. 408 base tiles x 3 variants = 1224 files.

The real tiles matter in this notebook for three things and no others: the **KID/FID reference
set**, the **memorization baseline**, and the **layout-fidelity ceiling**. Nothing here trains, so
no augmentation and no loader shuffling.

In [ ]:
def find_data_root(path):
    if (Path(path) / "images").is_dir(): return str(path)
    hits = [p for p in Path("/kaggle/input").glob("**/satellite_dataset") if (p / "images").is_dir()]
    assert hits, "satellite_dataset/ not found under /kaggle/input -- attach sugam24/satellite-datasets"
    print(f"DATA_ROOT auto-detected: {hits[0]}")
    return str(hits[0])

cfg.DATA_ROOT = find_data_root(cfg.DATA_ROOT)
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axis
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

h, w = cv2.imread(str(MASK_DIR / records[0][3]), cv2.IMREAD_COLOR).shape[:2]
NATIVE_GSD = 0.300                                        # m/px, recovered Aug 2026 (see the paper)
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print(f"native tile {w}x{h} @ {NATIVE_GSD:.3f} m/px = {w*NATIVE_GSD:.1f} x {h*NATIVE_GSD:.1f} m")
print(f"working GSD at {cfg.IMG_SIZE}: {w*NATIVE_GSD/cfg.IMG_SIZE:.3f} m/px "
      f"| at {cfg.GEN_SIZE}: {w*NATIVE_GSD/cfg.GEN_SIZE:.3f} m/px")

## 5. Mask colour diagnostic — run this before anything else

Same guard as the part-1 notebooks and both upstream stages. It matters here for a reason specific
to Stage 3: the LUT built in this cell defines the class indices that the **KID reference set**, the
**memorization baseline** and the **layout-fidelity ceiling** are all computed against. If
Residential and River were swapped by a channel-order mistake, the fakes would be scored against a
mislabelled ground truth and every number in section 19 would be quietly wrong while looking
perfectly reasonable.

The two asserts are the same two: Residential must dominate River by at least 3x, and the unmapped
share must be exactly zero.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}
IGNORE_BGR = (0, 0, 0)

# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.
def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

def _k(bgr): return int(pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0])

LUT   = np.full(1 << 24, cfg.IGNORE_INDEX, dtype=np.uint8)   # 16 MB; default = ignore
KNOWN = np.zeros(1 << 24, dtype=bool)                        # 16 MB; True only for palette colours
for bgr, idx in BGR_TO_CLASS.items():
    LUT[_k(bgr)] = idx; KNOWN[_k(bgr)] = True
KNOWN[_k(IGNORE_BGR)] = True

def rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:
    return LUT[pack_bgr(bgr_img)]

# ---- diagnostic over a random sample of raw full-res masks ----
rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 12, replace=False)]
# 12, not 40: each mask is a 5.5 MP PNG and this check costs ~9 s per file. The palette
# is uniform across the dataset, so 12 is plenty to catch a channel-order inversion, and
# the repack in section 6 verifies the LUT over all 1224 files anyway.
colour_hits, class_px, unmapped, total = {}, np.zeros(7, dtype=np.int64), 0, 0

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    keys = pack_bgr(m)
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = LUT[keys]
    class_px += np.bincount(np.where(idx == cfg.IGNORE_INDEX, 6, idx).ravel(), minlength=7)
    unmapped += int((~KNOWN[keys]).sum()); total += keys.size

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag_c = (f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known
             else ("-> IGNORE (palette black)" if c == IGNORE_BGR else "-> UNMAPPED -> ignore"))
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag_c}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(CODE_NAMES):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")
print(f"\nunmapped pixel share: {100*unmapped/total:.6f}%")

# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.
assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before sampling.")
assert unmapped == 0, f"{unmapped} pixels ({100*unmapped/total:.6f}%) are off-palette -- fix the masks first"
print("channel-order check PASSED | LUT is exact")

## 6. One-time repack to a 512 cache (+ a 128 label cache)

Byte-identical to the part-1 and Stage-2 repack, so if you already built that cache for the U-Net,
SegFormer or Stage-2 runs, set `PREBUILT_CACHE` to it and the 512 half of this cell is a no-op.

Your tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs to build a KID reference
set and a memorization index is pure waste; the repack pays for itself on the first pass.

Two sizes, as in Stage 1:

- **512** is the interface resolution. Real images at 512 are the KID/FID reference and the
  memorization index; real labels at 512 are the layout-fidelity ceiling and the condition-source
  ablation input.
- **128** is where mask-space statistics live, because that is the resolution Stage 1 actually
  models. Comparing a generated mask's fragmentation at 512 against a real mask's at 512 would
  measure the `INTER_NEAREST` upsample, not the generator.

Downsampling: `INTER_AREA` for images (correct for photometric downsampling), `INTER_NEAREST` for
labels applied **after** the LUT at full resolution (any interpolating kernel would invent classes
along every boundary).

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE  = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    keys = pack_bgr(msk)
    lab  = LUT[keys]                                            # full-res -> class index
    bad  = int((~KNOWN[keys]).sum())
    lab  = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == cfg.IGNORE_INDEX, 6, lab).ravel(), minlength=7), bad, keys.size

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts, bad_px, tot_px = {}, 0, 0
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, (b, bad, n) in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist(); bad_px += bad; tot_px += n
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  "
                      f"eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts, "size": cfg.IMG_SIZE,
               "unmapped_px": bad_px, "total_px": tot_px,
               "unmapped_pct": 100.0 * bad_px / max(tot_px, 1)}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

meta = json.load(open(MARKER))
PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in meta["counts"].items()}
print(f"LUT unmapped over the FULL dataset: {meta.get('unmapped_pct', float('nan')):.6f}% "
      f"({meta.get('unmapped_px', 0)} px)")
assert meta.get("unmapped_pct", 0.0) == 0.0, "off-palette pixels present in the full dataset"
for sub in ("images", "labels"):
    print(f"  {sub}: {len(list((CACHE / sub).glob('*.png')))} files")
print(f"cache size: {sum(f.stat().st_size for f in CACHE.rglob('*.png'))/1e6:.1f} MB")

## 7. Load the fold-1 tiles into RAM

Everything downstream reads these repeatedly — the KID reference is re-featurised per R set, the
memorization index is scanned once per fake — so holding them resident removes disk I/O from every
loop that matters.

Budget: 1224 x (512x512x3 + 512x512) uint8 is ~1.0 GB, comfortably inside Kaggle's ~13 GB, and the
128 label copies add 20 MB. The images are the expensive part, and note that Stage 2 could free
them after building its latent cache — this notebook cannot, because the real images *are* the KID
reference and the memorization index.

The 128 labels are produced here by `INTER_NEAREST` from the 512 cache rather than by a second full-
resolution pass. That is exactly what Stage 1 did, so the real-mask statistics in section 19 are
comparable to the ones in Stage 1's paper table without recomputation.

In [ ]:
RAM_IMG, RAM_LAB, RAM_L128 = {}, {}, {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)          # BGR
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        RAM_IMG[iname] = img
        RAM_LAB[iname] = lab
        RAM_L128[iname] = cv2.resize(lab, (cfg.GEN_SIZE, cfg.GEN_SIZE),
                                     interpolation=cv2.INTER_NEAREST)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = (sum(a.nbytes for a in RAM_IMG.values()) + sum(a.nbytes for a in RAM_LAB.values())
          + sum(a.nbytes for a in RAM_L128.values())) / 1e6
    print(f"RAM cache: {len(RAM_IMG)} tiles, {mb:.0f} MB, {time.time()-t0:.0f}s")
else:
    print("PRELOAD_RAM off -- every evaluation pass will hit the disk")

def real_img(iname):    # BGR uint8. .get() + fallback so section 13 can free RAM_IMG safely.
    a = RAM_IMG.get(iname) if cfg.PRELOAD_RAM else None
    if a is None: a = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
    return a
def real_lab(iname, mname):
    return RAM_LAB[iname] if cfg.PRELOAD_RAM else cv2.imread(str(CACHE / "labels" / mname),
                                                             cv2.IMREAD_GRAYSCALE)
def real_l128(iname, mname):
    if cfg.PRELOAD_RAM: return RAM_L128[iname]
    return cv2.resize(real_lab(iname, mname), (cfg.GEN_SIZE, cfg.GEN_SIZE),
                      interpolation=cv2.INTER_NEAREST)

## 8. Splitter — fold 1 of the sequence-block CV, and the three-way SHA1 agreement

`block_split` is **copied verbatim** from the part-1 notebooks and from both upstream stages. It
must be, because four notebooks now run the identical function and all four have to agree on which
tiles exist. Do not reimplement it, do not tidy it, do not change the buffer.

The `train_sha1` printed here is the SHA1 of the sorted train id list. It must match what Stage 1
and Stage 2 printed. Set `UPSTREAM_TRAIN_SHA1` in section 2 to turn that from a thing you eyeball
into an assert. If it disagrees, **stop**: it means one of the three notebooks is on a different
fold or a different buffer, and every fake pair this notebook produces would be contaminated by
tiles that Stage 4's validation block is about to score on.

`n_real = len(tr_recs)` is what the R ladder multiplies. For block fold 1 that is 268 base tiles x 3
variants = **804**.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)

def expand(ids): return [r for i in ids for r in groups[i]]
tr_recs, va_recs = expand(tr_ids), expand(va_ids)
ORIG_VAL = [i for i, r in enumerate(va_recs) if r[1] == "orig"]     # hf/vf are mirrors, not scenes

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
buffered = sorted(set(BASE_IDS) - set(tr_ids) - set(va_ids))
split_sha1 = hashlib.sha1(",".join(map(str, sorted(tr_ids))).encode()).hexdigest()

SPLIT = {"split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "n_folds": cfg.N_FOLDS,
         "block_buffer": cfg.BLOCK_BUFFER,
         "rule": "block_split(), verbatim from the part-1 notebooks",
         "train_ids": sorted(tr_ids), "val_ids": sorted(va_ids), "buffered_ids": buffered,
         "train_sha1": split_sha1}
json.dump(SPLIT, open(f"{cfg.OUT_DIR}/fold{cfg.FOLD}_split.json", "w"), indent=2)

N_REAL = len(tr_recs)
R_SIZES = {r: int(r * N_REAL) for r in cfg.R_MULTIPLIERS}
TOTAL_PAIRS = sum(R_SIZES.values()) if cfg.INDEPENDENT_DRAWS else max(R_SIZES.values())

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "
      f"({len(buffered)} buffered out: {buffered})")
print(f"  -> {len(tr_recs)} train pairs / {len(va_recs)} val pairs "
      f"({len(ORIG_VAL)} orig-variant val scenes)")
print(f"  train_sha1: {split_sha1}   <-- must match Stage 1 AND Stage 2")
if cfg.UPSTREAM_TRAIN_SHA1:
    assert split_sha1 == cfg.UPSTREAM_TRAIN_SHA1, (
        f"split disagreement with the upstream stages\n  upstream: {cfg.UPSTREAM_TRAIN_SHA1}"
        f"\n  stage3  : {split_sha1}")
    print("  three-way agreement: ASSERTED")
else:
    print("  set cfg.UPSTREAM_TRAIN_SHA1 to assert this instead of eyeballing it")

print(f"\nR ladder against n_real = {N_REAL}:")
for r in cfg.R_MULTIPLIERS:
    print(f"  R{r} = {r}x -> {R_SIZES[r]} fake pairs")
print(f"  total to generate: {TOTAL_PAIRS} pairs "
      f"({'independent draws' if cfg.INDEPENDENT_DRAWS else 'nested, R3 only'})")
print(f"  saved -> {cfg.OUT_DIR}/fold{cfg.FOLD}_split.json")

## 9. Stage-1 generator — loaded frozen from the Hub, with its own sampler

This is Stage 1's section-22 standalone loader, moved here and extended. The UNet is rebuilt from
the **full `UNet2DModel` config embedded in `best.pt`**, so the architecture is reconstructed
exactly as trained rather than re-derived from a CFG that might have drifted.

Three things this cell hard-checks before it will sample anything:

1. **`LABEL_CODING == "onehot"`.** The run-A binary-code checkpoint decodes differently (threshold,
   then nearest-neighbour assignment to valid codes). Feeding it to an argmax decoder produces
   plausible-looking garbage, which is the worst kind.
2. **zero terminal SNR is paired with v-prediction.** Stage 1's section 11 refuses the broken
   combination at training time; the same refusal is repeated here because the scheduler kwargs are
   rebuilt from the checkpoint's `cfg` and a mismatch would silently change the sampling
   trajectory.
3. **the DDIM scheduler kwargs match the training betas exactly** — `linear`, `beta_start`,
   `beta_end`, `num_train_timesteps` all read from the checkpoint, never retyped.

`ambiguity` is carried over from Stage 1: the fraction of pixels whose top two one-hot channels are
within `AMB_MARGIN` of each other. It is the honest measure of how far the samples are from settling
on the code corners, and it is what boundary speckle looks like before the modal filter hides it.

In [ ]:
from diffusers import UNet2DModel, DDIMScheduler
from huggingface_hub import hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

def load_stage1(repo_id, token=None, device="cuda"):
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location=device)
    c  = ck["cfg"]
    assert c.get("LABEL_CODING", "onehot") == "onehot", (
        "this sampler decodes one-hot by argmax; the binary-code arm needs run A's decoder")
    if c.get("ZERO_TERMINAL_SNR", False):
        assert c["PREDICTION_TYPE"] == "v_prediction", (
            "zero terminal SNR with epsilon prediction is the broken combination Stage 1 refuses")
    m = UNet2DModel.from_config(ck["model_config"]).to(device).eval()
    m.load_state_dict(ck["model_state"])
    m.requires_grad_(False)
    return m, c, ck

def s1_scheduler(c, steps):
    kw = dict(num_train_timesteps=c["TIMESTEPS"], beta_start=c["BETA_START"],
              beta_end=c["BETA_END"], beta_schedule="linear",
              prediction_type=c["PREDICTION_TYPE"], clip_sample=True, steps_offset=0)
    if c.get("ZERO_TERMINAL_SNR", False):
        kw.update(rescale_betas_zero_snr=True, timestep_spacing="trailing", set_alpha_to_one=True)
    else:
        kw.update(set_alpha_to_one=False)
    sc = DDIMScheduler(**kw); sc.set_timesteps(steps)
    return sc

@torch.no_grad()
def sample_masks(m, c, n, steps, seed, batch=None, progress=False):
    # returns (N, GEN_SIZE, GEN_SIZE) uint8 in codes {0..6}, plus the ambiguity rate
    batch = batch or cfg.S1_BATCH
    nch, gs = c["N_CH"], c["GEN_SIZE"]
    out, amb_n, amb_d = [], 0, 0
    for s in range(0, n, batch):
        b = min(batch, n - s)
        gg = torch.Generator(); gg.manual_seed(seed + s)
        x = torch.randn(b, nch, gs, gs, generator=gg).cuda()
        sc = s1_scheduler(c, steps)
        for t in sc.timesteps:
            with torch.cuda.amp.autocast(enabled=cfg.AMP):
                pred = m(x, t).sample
            x = sc.step(pred.float(), t, x, eta=0.0).prev_sample
        x = x.clamp(-1, 1)
        top2 = x.topk(2, dim=1).values
        amb_n += int(((top2[:, 0] - top2[:, 1]) < cfg.AMB_MARGIN).sum()); amb_d += b * gs * gs
        out.append(x.argmax(dim=1).to(torch.uint8).cpu().numpy())
        if progress and (s // batch) % 10 == 0:
            print(f"    masks {min(s+b, n)}/{n}", flush=True)
    return np.concatenate(out), amb_n / max(amb_d, 1)

S1_MODEL, S1_CFG, S1_CK = load_stage1(cfg.S1_REPO, token=HF_TOKEN)
S1_STEPS = cfg.S1_DDIM_STEPS or S1_CFG.get("FINAL_DDIM_STEPS", 200)
assert S1_CFG["GEN_SIZE"] == cfg.GEN_SIZE and S1_CFG["OUT_SIZE"] == cfg.IMG_SIZE, \
    "Stage 1 was trained at a different resolution than this notebook assumes"
assert S1_CFG["FOLD"] == cfg.FOLD and S1_CFG["SPLIT_MODE"] == cfg.SPLIT_MODE, \
    f"Stage 1 checkpoint is {S1_CFG['SPLIT_MODE']} fold {S1_CFG['FOLD']}, not fold {cfg.FOLD}"

print(f"Stage 1 loaded: {cfg.S1_REPO}")
print(f"  epoch {S1_CK.get('epoch')} | selected on {S1_CK.get('select_on')} = "
      f"{S1_CK.get('best_score', float('nan')):.5f} | arch {S1_CK.get('arch')}")
print(f"  coding {S1_CFG['LABEL_CODING']} {S1_CFG['N_CH']}ch | pred {S1_CFG['PREDICTION_TYPE']} | "
      f"zsnr {S1_CFG.get('ZERO_TERMINAL_SNR')} | sampling at {S1_STEPS} DDIM steps")
print(f"  params {sum(p.numel() for p in S1_MODEL.parameters())/1e6:.1f} M")

_t0 = time.time()
_demo, _amb = sample_masks(S1_MODEL, S1_CFG, 8, steps=16, seed=99)
print(f"  smoke sample: {_demo.shape} codes {sorted(np.unique(_demo).tolist())} "
      f"in {time.time()-_t0:.1f}s at 16 steps (16 is for the smoke test only)")

## 10. Mask post-processing and distribution-matched selection

Copied from Stage 1's section 13 and used here for the same reason it existed there: the raw
samples carry pixel-level boundary noise that the source polygons never had, and the raw pool's
class marginal drifts from the real one.

`smooth_mask` does four things in order, and the order is the whole point:

1. Lift Road and River out and refill their footprint from the nearest bulk label, so the modal
   filter cannot erase a 2 px-wide line. A thin class is always the minority in a 3x3 window;
   running the filter on the full mask deletes every road.
2. Modal-filter the bulk layer, with a tie-break toward the incumbent label so the filter cannot
   drift the class marginal.
3. Per thin class: binary **closing** to bridge gaps, then drop components below `min_px_thin`.
   Closing is what fragmentation needs — deletion alone can only ever make it worse.
4. Paint Road back, then River, so a river crossing a road stays a river. That matches how the
   source polygons were digitized.

`greedy_match` then picks the kept subset so the *selected* set's class marginal is closest to the
real one **in KL**. Minimising L1 instead would happily drop River entirely; KL punishes that and L1
does not.

### The sweep runs once, not per R set

`pp_sweep` is run in section 11 on the R3 pool only, and the winning knobs are then applied
unchanged to all three sets. Sweeping per set would give each R its own post-processing, and the
Stage-4 curve would then confound "more data" with "different smoothing".

### Declare this in the paper

The hand-off masks are post-processed. Report the **unfiltered** class-share KL as the generator's
own score and the filtered one as the hand-off set's, exactly as Stage 1 did.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import ndimage as ndi

STRUCT8 = np.ones((3, 3), dtype=int)          # 8-connectivity: correct for linear features

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlib
PALETTE_RGB[cfg.IGNORE_CODE] = (0, 0, 0)               # code 6, inside the models
PALETTE_RGB[cfg.IGNORE_INDEX] = (0, 0, 0)              # 255, on disk

def to_code(lab):
    return np.where(lab == cfg.IGNORE_INDEX, cfg.IGNORE_CODE, lab).astype(np.uint8)

def colorize(lab): return PALETTE_RGB[to_code(lab)]

LEGEND = ([Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)] +
          [Patch(color="black", label="ignore/unannotated")])

def mask_grid(stack, ncol=4, title=None, cell=2.1):
    n = len(stack); nrow = int(math.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * cell, nrow * cell))
    axes = np.atleast_1d(axes).ravel()
    for a in axes: a.axis("off")
    for i, m in enumerate(stack): axes[i].imshow(colorize(m))
    if title: fig.suptitle(title, fontsize=11)
    fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    return fig

# ---- distribution ----
def class_shares(stack):
    h = np.bincount(to_code(np.asarray(stack)).ravel(), minlength=cfg.N_CODES).astype(np.float64)
    return h / max(h.sum(), 1.0)

def kl_div(p, q, eps=1e-8):
    p = np.clip(p, eps, None); p = p / p.sum()
    q = np.clip(q, eps, None); q = q / q.sum()
    return float((p * np.log(p / q)).sum())

# ---- landscape ----
LS_KEYS = ["n_patches", "mean_area", "largest_share", "edge_density", "share"]

def landscape_one(a):
    a = to_code(a); out, area = {}, a.size
    for c in range(cfg.N_CODES):
        m = a == c; npx = int(m.sum())
        if npx == 0:
            out[c] = dict(n_patches=0.0, mean_area=0.0, largest_share=0.0,
                          edge_density=0.0, share=0.0); continue
        lab, n = ndi.label(m, structure=STRUCT8)
        ar = np.bincount(lab.ravel())[1:]
        eh = int(((a[:, 1:] == c) ^ (a[:, :-1] == c)).sum())      # vertical class boundaries
        ev = int(((a[1:, :] == c) ^ (a[:-1, :] == c)).sum())      # horizontal ones
        out[c] = dict(n_patches=float(n), mean_area=float(ar.mean()),
                      largest_share=float(ar.max() / npx),
                      edge_density=1000.0 * (eh + ev) / area, share=npx / area)
    p = np.bincount(a.ravel(), minlength=cfg.N_CODES) / area
    p = p[p > 0]
    return out, float(-(p * np.log(p)).sum())                     # Shannon diversity (SHDI)

def landscape_table(stack, name):
    acc = {c: {k: [] for k in LS_KEYS} for c in range(cfg.N_CODES)}; shdi = []
    for a in stack:
        per, s = landscape_one(a); shdi.append(s)
        for c, d in per.items():
            for k in LS_KEYS: acc[c][k].append(d[k])
    rows = []
    for c in range(cfg.N_CODES):
        r = {"code": c, "cls": CODE_NAMES[c], "set": name}
        for k in LS_KEYS:
            v = np.asarray(acc[c][k], dtype=np.float64)
            r[f"{k}_mean"] = float(v.mean()); r[f"{k}_sd"] = float(v.std())
        rows.append(r)
    return pd.DataFrame(rows), np.asarray(shdi)

# ---- connectivity of the thin classes ----
THIN = {1: "Road", 2: "River"}

def connectivity_stats(stack, c):
    rows = []
    for a in stack:
        a = to_code(a); m = a == c; npx = int(m.sum())
        if npx == 0:
            rows.append(dict(present=0, n_comp=0, frag=np.nan, mean_area=np.nan,
                             largest_share=np.nan, share=0.0)); continue
        lab, n = ndi.label(m, structure=STRUCT8)
        ar = np.bincount(lab.ravel())[1:]
        rows.append(dict(present=1, n_comp=int(n), frag=1000.0 * n / npx,
                         mean_area=float(ar.mean()), largest_share=float(ar.max() / npx),
                         share=npx / a.size))
    return pd.DataFrame(rows)

def presence_rate(stack):
    # fraction of masks in which each class appears at all -- NOT the same as the pooled share.
    # A pool can post the right pooled River share while 95% of individual masks contain no river,
    # and only the per-mask rate exposes that.
    P = np.zeros(cfg.N_CODES)
    for a in stack:
        u = np.unique(to_code(a))
        P[u] += 1
    return P / max(len(stack), 1)

# ---- post-processing: despeckle, then distribution-matched subset selection ----
def smooth_mask(a, modal_k=0, close_k=0, min_px_thin=0, min_px_bulk=0, thin=(1, 2)):
    a = to_code(a)
    tm = {c: (a == c) for c in thin}
    any_thin = np.zeros_like(a, bool)
    for m in tm.values(): any_thin |= m

    bulk = a.copy()
    if any_thin.any() and (~any_thin).any():
        _, ix = ndi.distance_transform_edt(any_thin, return_distances=True, return_indices=True)
        bulk[any_thin] = bulk[tuple(i[any_thin] for i in ix)]

    if modal_k and modal_k > 1:
        k = np.ones((modal_k, modal_k), np.float32) / (modal_k * modal_k)
        votes = np.stack([ndi.convolve((bulk == c).astype(np.float32), k, mode="nearest")
                          for c in range(cfg.N_CODES)])
        # tie-break toward the incumbent label so the filter cannot drift the class marginal
        rr, cc = np.arange(bulk.shape[0])[:, None], np.arange(bulk.shape[1])[None]
        votes[bulk, rr, cc] += 1e-3
        bulk = votes.argmax(0).astype(np.uint8)

    if min_px_bulk:
        for c in range(cfg.N_CODES):
            if c in thin: continue
            m = bulk == c
            if not m.any(): continue
            lab, _ = ndi.label(m, structure=STRUCT8)
            ar = np.bincount(lab.ravel()); ar[0] = 0
            small = np.where((ar > 0) & (ar < min_px_bulk))[0]
            if len(small):
                drop = np.isin(lab, small)
                if (~drop).any():
                    _, ix = ndi.distance_transform_edt(drop, return_distances=True,
                                                       return_indices=True)
                    bulk[drop] = bulk[tuple(i[drop] for i in ix)]

    out = bulk
    for c in thin:                                   # Road first, River last
        m = tm[c]
        if not m.any(): continue
        if close_k and close_k > 1:
            m = ndi.binary_closing(m, structure=np.ones((close_k, close_k), bool))
        if min_px_thin:
            lab, _ = ndi.label(m, structure=STRUCT8)
            ar = np.bincount(lab.ravel()); ar[0] = 0
            m = np.isin(lab, np.where(ar >= min_px_thin)[0])
        out = np.where(m, c, out).astype(np.uint8)
    return out

def frag_of(stack, c):
    v = [1000.0 * ndi.label(to_code(a) == c, structure=STRUCT8)[1] / (to_code(a) == c).sum()
         for a in stack if (to_code(a) == c).any()]
    return float(np.mean(v)) if v else float("nan")

def pp_report(stack, ref):
    p_r, p_g = class_shares(ref), class_shares(stack)
    d = {"kl": kl_div(p_r, p_g)}
    for c, nm in [(1, "Road"), (2, "River")]:
        fr, fg = frag_of(ref, c), frag_of(stack, c)
        d[f"frag_{nm}"]  = fg / fr if fr else float("nan")
        d[f"share_{nm}"] = p_g[c] / p_r[c] if p_r[c] else float("nan")
    d["npatch"] = float(np.mean([
        sum(ndi.label(to_code(a) == c, structure=STRUCT8)[1]
            for c in range(cfg.N_CODES) if (to_code(a) == c).any())
        for a in stack]))
    sh = [np.bincount(to_code(a).ravel(), minlength=cfg.N_CODES) / a.size for a in stack]
    d["shdi"] = float(np.mean([-(q[q > 0] * np.log(q[q > 0])).sum() for q in sh]))
    return d

def pp_sweep(pool, ref, n=None):
    n = cfg.PP_SWEEP_N if n is None else n
    sub = pool[np.random.RandomState(0).choice(len(pool), min(n, len(pool)), replace=False)]
    base, tgt = pp_report(sub, ref), pp_report(ref, ref)
    hdr = (f"{'config':<32}{'KL':>8}{'fragRoad':>10}{'fragRiv':>9}{'shRoad':>8}"
           f"{'shRiv':>8}{'npatch':>8}{'SHDI':>7}")
    print("\n--- post-processing sweep (R3 pool vs real fold-1 train masks) ---"); print(hdr)
    print(f"{'REAL (target)':<32}{0.0:8.5f}{1.0:10.2f}{1.0:9.2f}{1.0:8.2f}{1.0:8.2f}"
          f"{tgt['npatch']:8.1f}{tgt['shdi']:7.3f}")
    print(f"{'raw pool (no post-processing)':<32}{base['kl']:8.5f}{base['frag_Road']:10.2f}"
          f"{base['frag_River']:9.2f}{base['share_Road']:8.2f}{base['share_River']:8.2f}"
          f"{base['npatch']:8.1f}{base['shdi']:7.3f}")
    print("-" * len(hdr))
    res = []
    for mk in (0, 3, 5):
        for ck in (0, 3):
            for mp in (12, 30, 60):
                for mb in (0, 40):
                    st = np.stack([smooth_mask(a, mk, ck, mp, mb) for a in sub])
                    r = pp_report(st, ref); r["cfg"] = (mk, ck, mp, mb); res.append(r)
                    print(f"{f'modal{mk} close{ck} thin{mp} bulk{mb}':<32}{r['kl']:8.5f}"
                          f"{r['frag_Road']:10.2f}{r['frag_River']:9.2f}{r['share_Road']:8.2f}"
                          f"{r['share_River']:8.2f}{r['npatch']:8.1f}{r['shdi']:7.3f}", flush=True)
    ok = [r for r in res
          if r["frag_Road"] <= 3.0 and r["frag_River"] <= 3.0
          and 0.25 <= r["share_Road"] <= 2.5 and 0.25 <= r["share_River"] <= 2.5
          and r["kl"] <= 2 * base["kl"]]
    print("-" * len(hdr))
    if ok:
        b = min(ok, key=lambda r: abs(np.log(r["npatch"] / tgt["npatch"]))
                                  + abs(np.log(r["shdi"] / tgt["shdi"])))
        print(f"{len(ok)}/{len(res)} configs pass both thin-class gates. "
              f"CHOSEN: modal{b['cfg'][0]} close{b['cfg'][1]} thin{b['cfg'][2]} bulk{b['cfg'][3]}"
              f"  (River frag {b['frag_River']:.2f}, was {base['frag_River']:.2f})")
        return b["cfg"]
    print("NO config passes both gates. River fragmentation is NOT a post-processing problem:")
    print("  at 128 px a river is ~3 px wide and the model cannot hold a continuous channel.")
    print("  This is a real negative result -- report the FAIL and put the resolution limit in")
    print("  the discussion. Falling back to (0, 3, 30, 0) for the hand-off sets.")
    return (0, 3, 30, 0)

def greedy_match(stack, target, k):
    # Pick k of the pool so the SELECTED set's class marginal is closest to the real one in KL.
    # Minimising L1 instead would happily drop River entirely -- KL punishes that, L1 does not.
    sh = np.stack([np.bincount(to_code(a).ravel(), minlength=cfg.N_CODES) / a.size for a in stack])
    t = np.clip(target, 1e-8, None); t = t / t.sum()
    avail, chosen, cur = list(range(len(stack))), [], np.zeros(cfg.N_CODES)
    for step in range(min(k, len(stack))):
        cand = (cur * step + sh[avail]) / (step + 1)
        c = np.clip(cand, 1e-8, None); c = c / c.sum(1, keepdims=True)
        j = int((t[None] * np.log(t[None] / c)).sum(1).argmin())
        cur = cand[j]; chosen.append(avail.pop(j))
    return chosen, cur

def upsample_mask(m128):
    # 128 codes {0..6} -> 512 on-disk values {0..5, 255}. INTER_NEAREST is exact for label maps;
    # any other kernel invents classes along every boundary.
    # Defined HERE, not in section 17, because sections 12 and 14 also need it: the Stage-2
    # control branch downsamples its conditioning by 8, so a 128 px mask produces a 16x16
    # condition against a 64x64 latent and the addition inside ControlNet fails.
    big = cv2.resize(to_code(m128), (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return np.where(big == cfg.IGNORE_CODE, cfg.IGNORE_INDEX, big).astype(np.uint8)

# ---- the real reference stacks, at both resolutions ----
TRAIN_128 = np.stack([real_l128(r[2], r[3]) for r in tr_recs])
VAL_128   = np.stack([real_l128(va_recs[i][2], va_recs[i][3]) for i in ORIG_VAL])
P_REAL    = class_shares(TRAIN_128)

print(f"real reference: TRAIN_128 {TRAIN_128.shape} | VAL_128 {VAL_128.shape}")
print("real fold-1 train class shares (%):",
      {CODE_NAMES[i]: round(100 * P_REAL[i], 3) for i in range(cfg.N_CODES)})
print("real per-mask presence rate (%):",
      {CODE_NAMES[i]: round(100 * presence_rate(TRAIN_128)[i], 1) for i in range(cfg.N_CODES)})
_f = mask_grid(TRAIN_128[:8], ncol=4, title=f"real fold-{cfg.FOLD} train masks (128, class indices)")
plt.show()
print("mask post-processing module ready | thin classes:", THIN)

## 11. The mask pools — one per R set, cached and pushed

For each R the pipeline is: sample `ceil(R_SIZE * MASK_POOL_RATIO)` masks at the Stage-1
checkpoint's own `FINAL_DDIM_STEPS`, post-process the whole pool with the knobs chosen once by the
sweep, then `greedy_match` down to exactly `R_SIZE`.

**Seeds.** Each R set gets `MASK_SEED_BASE + 100000*R`, so the three pools are disjoint draws rather
than three windows onto one. If `INDEPENDENT_DRAWS` is off, R1 and R2 become prefixes of R3 and only
one pool is drawn — cheaper, but then the Stage-4 curve is not a curve, and you must say so.

**Caching.** Each finished pool is written to `masks_R{r}.npy` and pushed to the Hub. A resumed
session reloads it rather than resampling, which is what makes the image half of this notebook
resumable at all: repainting a *different* mask at index 1400 after a crash would silently
half-corrupt the set.

Mask sampling is cheap — a 12 M-parameter UNet at 128 px — so this whole cell is roughly 30 minutes
even at 200 DDIM steps for 12 000 masks. The expensive half is section 18.

In [ ]:
def _mask_cache_path(r): return f"{cfg.OUT_DIR}/masks_R{r}_{tag}.npy"

PP_DEFAULT = (0, 3, 30, 0)
PP_PATH = f"{cfg.OUT_DIR}/pp_knobs_{tag}.json"

def build_mask_pool(r, pp_knobs, sweep_here=False):
    keep = R_SIZES[r]
    pool_n = int(math.ceil(keep * cfg.MASK_POOL_RATIO)) if cfg.FILTER_MASKS else keep
    seed = cfg.MASK_SEED_BASE + 100000 * r
    print(f"\n=== R{r}: sampling {pool_n} masks -> keeping {keep} (seed {seed}) ===", flush=True)
    t0 = time.time()
    RAW, amb = sample_masks(S1_MODEL, S1_CFG, pool_n, steps=S1_STEPS, seed=seed, progress=True)
    print(f"  sampled in {(time.time()-t0)/60:.1f} min | ambiguity {amb:.4%}")
    if amb > 0.02:
        print("  WARNING: >2% of pixels have their top two channels nearly tied -- expect "
              "boundary speckle that the modal filter will be doing real work to hide")
    if sweep_here and cfg.PP_SWEEP:
        pp_knobs = pp_sweep(RAW, TRAIN_128)
        json.dump(list(pp_knobs), open(PP_PATH, "w"))        # so a resumed session reuses them
    if pp_knobs is None:
        pp_knobs = PP_DEFAULT
        print(f"  no sweep and no pinned knobs -- falling back to {PP_DEFAULT}")
    if not cfg.FILTER_MASKS:
        return RAW, RAW, pp_knobs, amb
    CLEAN = np.stack([smooth_mask(a, *pp_knobs) for a in RAW])
    sel, _ = greedy_match(CLEAN, P_REAL, keep)
    SEL = CLEAN[np.array(sorted(sel))]
    print(f"  filter: pool {len(RAW)} -> {len(SEL)} | "
          f"KL(real||gen) raw {kl_div(P_REAL, class_shares(RAW)):.5f} -> "
          f"{kl_div(P_REAL, class_shares(SEL)):.5f}")
    return SEL, RAW, pp_knobs, amb

PP_KNOBS = None
if all(v is not None for v in (cfg.PP_MODAL, cfg.PP_CLOSE, cfg.PP_THIN, cfg.PP_BULK)):
    PP_KNOBS = (cfg.PP_MODAL, cfg.PP_CLOSE, cfg.PP_THIN, cfg.PP_BULK)
    print("post-processing knobs pinned in CFG:", PP_KNOBS, "-- the sweep will not run")
elif os.path.exists(PP_PATH):                       # a previous session already swept
    PP_KNOBS = tuple(json.load(open(PP_PATH)))
    print("post-processing knobs restored from an earlier session:", PP_KNOBS)
else:
    _dl = None
    try: _dl = hf_hub_download(HF_REPO, os.path.basename(PP_PATH), token=HF_TOKEN)
    except Exception: pass
    if _dl:
        shutil.copy(_dl, PP_PATH); PP_KNOBS = tuple(json.load(open(PP_PATH)))
        print(f"post-processing knobs restored from {HF_REPO}:", PP_KNOBS)

# ---- push each pool the moment it exists, not at section 15 ----------------------------
# Mask sampling is the single most expensive thing in this notebook (~7.8 h for 12060 masks at
# 200 DDIM steps). Deferring the upload to section 15 means any failure in section 12, 13 or 14
# takes those hours with it. Push here instead, immediately, before anything else can break.
from huggingface_hub import HfApi, create_repo
_api = HfApi(token=HF_TOKEN)
try:
    create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)
except Exception as e:
    print("create_repo failed (pushes will be attempted anyway):", type(e).__name__, e)

def push_pool(path):
    try:
        _api.upload_file(path_or_fileobj=path, path_in_repo=os.path.basename(path),
                         repo_id=HF_REPO, token=HF_TOKEN,
                         commit_message=f"mask pool {os.path.basename(path)}")
        print(f"  PUSHED {os.path.basename(path)} -> {HF_REPO} "
              f"({os.path.getsize(path)/1e6:.1f} MB)")
        return True
    except Exception as e:
        print("  POOL PUSH FAILED -- this pool exists ONLY in /kaggle/working:",
              type(e).__name__, e)
        return False

def _restore_pool(r, dest):
    # 1. an attached dataset (e.g. the output of an earlier failed version)
    if cfg.MASK_RESTORE_DIR:
        cand = os.path.join(cfg.MASK_RESTORE_DIR, os.path.basename(dest))
        if os.path.exists(cand):
            shutil.copy(cand, dest); print(f"R{r}: mask pool restored from {cfg.MASK_RESTORE_DIR}")
            return True
    # 2. the Hub
    try:
        d = hf_hub_download(HF_REPO, os.path.basename(dest), token=HF_TOKEN)
        shutil.copy(d, dest); print(f"R{r}: mask pool restored from {HF_REPO}")
        return True
    except Exception:
        return False

MASKS128, MASK_META = {}, {}
R_ORDER = sorted(cfg.R_MULTIPLIERS, reverse=True)      # biggest first: it carries the sweep
for j, r in enumerate(R_ORDER):
    p = _mask_cache_path(r)
    if not os.path.exists(p):
        _restore_pool(r, p)
    if os.path.exists(p):
        MASKS128[r] = np.load(p)
        assert len(MASKS128[r]) == R_SIZES[r], \
            f"cached R{r} has {len(MASKS128[r])} masks, expected {R_SIZES[r]} -- delete {p}"
        MASK_META[r] = {"restored": True}
        print(f"R{r}: reusing cached mask pool {MASKS128[r].shape}")
        continue
    if not cfg.INDEPENDENT_DRAWS and j > 0:
        big = R_ORDER[0]
        MASKS128[r] = MASKS128[big][:R_SIZES[r]]
        MASK_META[r] = {"nested_prefix_of": big}
        print(f"R{r}: nested prefix of R{big}")
    else:
        SEL, RAW, PP_KNOBS, amb = build_mask_pool(r, PP_KNOBS, sweep_here=(PP_KNOBS is None))
        MASKS128[r] = SEL
        MASK_META[r] = {"ambiguity": float(amb), "pool_n": int(len(RAW)),
                        "kl_raw": kl_div(P_REAL, class_shares(RAW)),
                        "kl_selected": kl_div(P_REAL, class_shares(SEL)),
                        "pp_knobs": list(PP_KNOBS)}
    np.save(_mask_cache_path(r), MASKS128[r])
    push_pool(_mask_cache_path(r))                     # immediately, not at section 15
    if os.path.exists(PP_PATH): push_pool(PP_PATH)

PP_KNOBS = PP_KNOBS or PP_DEFAULT
json.dump(list(PP_KNOBS), open(PP_PATH, "w"))
print("\n--- mask pools ready ---")
print(f"post-processing applied to all sets: modal{PP_KNOBS[0]} close{PP_KNOBS[1]} "
      f"thin{PP_KNOBS[2]} bulk{PP_KNOBS[3]}")
_rows = []
for r in cfg.R_MULTIPLIERS:
    s = class_shares(MASKS128[r]); pr = presence_rate(MASKS128[r])
    _rows.append(dict(set=f"R{r}", n=len(MASKS128[r]),
                      **{f"{CODE_NAMES[i][:5]}%": round(100 * s[i], 3) for i in range(cfg.N_CODES)},
                      Road_in=round(100 * pr[1], 1), River_in=round(100 * pr[2], 1),
                      KL=round(kl_div(P_REAL, s), 5)))
_rows.append(dict(set="REAL", n=len(TRAIN_128),
                  **{f"{CODE_NAMES[i][:5]}%": round(100 * P_REAL[i], 3) for i in range(cfg.N_CODES)},
                  Road_in=round(100 * presence_rate(TRAIN_128)[1], 1),
                  River_in=round(100 * presence_rate(TRAIN_128)[2], 1), KL=0.0))
print(pd.DataFrame(_rows).to_string(index=False))
print("\n  Road_in / River_in are the PER-MASK presence rates. A pooled share that matches while")
print("  the presence rate collapses means most masks have no road or river at all, and Stage 4")
print("  cannot possibly gain on those classes. Read these two columns before anything else.")

del S1_MODEL; gc.collect(); torch.cuda.empty_cache()
print("\nStage-1 UNet released -- the image generator needs the VRAM")

## 12. Stage-2 generator — frozen SD1.5 + LoRA + trained control branch

Rebuilt exactly as Stage 2's section 12 built it, then loaded from `best.pt`:

| piece | state | source |
|---|---|---|
| SD 1.5 UNet | frozen, fp16 | Hub, `SD_MODEL_ID` from the checkpoint's cfg |
| LoRA on `to_q`/`to_k`/`to_v`/`to_out.0` | loaded, fp32 | `best.pt["lora_state"]` |
| control branch (conv_in swapped to 7 channels) | loaded, fp32 | `best.pt["control_state"]` |
| canny branch | frozen, fp16 | Hub, sampling-time only |
| VAE, CLIP text encoder | frozen | Hub |

Three details that are easy to get wrong and expensive to notice late:

- **The conditioning conv must be swapped before loading.** The pretrained seg ControlNet has a
  3-channel `conv_in`; the trained one has 7. Loading the state dict into an unswapped model raises
  on a shape mismatch, which is the good case. Swapping *after* loading would silently discard the
  trained conditioning weights and you would get texture with no layout.
- **`lora_state` and `control_state` are stored as fp16** (Stage 2 halves them on save). They are
  cast back to fp32 here; leaving them fp16 inside an fp32 module produces a dtype error deep in the
  first attention block.
- **Sampling hyper-parameters come from the checkpoint's cfg**, not from this notebook's CFG,
  unless you explicitly overrode them in section 2. The resolved values print below and are recorded
  in section 20's results JSON and in every manifest.

Captions are rebuilt from each mask's own class proportions by the same templated function Stage 2
trained with. Do not substitute a nicer prompt: the model was conditioned on this vocabulary, and a
caption outside it is an out-of-distribution text embedding.

In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, ControlNetModel
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig
import threading, queue

S2_CK  = torch.load(hf_hub_download(cfg.S2_REPO, "best.pt", token=HF_TOKEN), map_location="cpu")
S2_CFG = S2_CK["cfg"]
assert S2_CFG["COND_MODE"] == "onehot", \
    f"this notebook chains the one-hot control branch; checkpoint is {S2_CFG['COND_MODE']}"
assert S2_CFG["FOLD"] == cfg.FOLD and S2_CFG["SPLIT_MODE"] == cfg.SPLIT_MODE, \
    f"Stage 2 checkpoint is {S2_CFG['SPLIT_MODE']} fold {S2_CFG['FOLD']}, not fold {cfg.FOLD}"
assert S2_CFG["IMG_SIZE"] == cfg.IMG_SIZE, "Stage 2 was trained at a different resolution"

# ---- resolve the sampling hyper-parameters: CFG override, else inherit ----
def inherit(name, key=None, default=None):
    v = getattr(cfg, name)
    return v if v is not None else S2_CFG.get(key or name, default)

DDIM_STEPS  = int(inherit("DDIM_STEPS",  default=30))
GUIDANCE    = float(inherit("GUIDANCE",  default=7.5))
COND_SCALE  = float(inherit("COND_SCALE", default=1.0))
CANNY_SCALE = float(inherit("CANNY_SCALE", default=0.4))
USE_CANNY   = bool(inherit("USE_CANNY",  default=True))
COND_CH     = cfg.N_COND_CH

# ---- which GPUs ----
N_GPU = torch.cuda.device_count()
DEVICES = ([f"cuda:{i}" for i in range(N_GPU)] if (cfg.MULTI_GPU and N_GPU > 1) else ["cuda:0"])
CTRL_DTYPE = torch.float16 if cfg.S2_FP16 else torch.float32
print(f"{N_GPU} GPU(s) visible | sampling on {DEVICES} | control branch in {CTRL_DTYPE}")
if N_GPU > 1 and not cfg.MULTI_GPU:
    print("  MULTI_GPU is off -- you are leaving a whole T4 idle")

def build_engine(device):
  # Every allocation and kernel launch inside this block targets `device`. Without the context
  # manager, from_pretrained's intermediate buffers land on cuda:0 no matter what .to(device)
  # says afterwards.
  with torch.cuda.device(device):
      u = UNet2DConditionModel.from_pretrained(S2_CFG["SD_MODEL_ID"], subfolder="unet",
                                               torch_dtype=torch.float16).to(device)
      u.requires_grad_(False)

      c = ControlNetModel.from_pretrained(S2_CFG["CONTROLNET_ID"],
                                          torch_dtype=torch.float32).to(device)
      old = c.controlnet_cond_embedding.conv_in            # 3 -> 7 channels, BEFORE loading
      new = nn.Conv2d(COND_CH, old.out_channels, kernel_size=old.kernel_size,
                      stride=old.stride, padding=old.padding)
      c.controlnet_cond_embedding.conv_in = new.to(device, torch.float32)
      c.register_to_config(conditioning_channels=COND_CH)

      n_lora = 0
      if S2_CK.get("lora_state"):
          u.add_adapter(LoraConfig(r=S2_CFG["LORA_RANK"], lora_alpha=S2_CFG["LORA_ALPHA"],
                                   lora_dropout=0.0, init_lora_weights="gaussian",
                                   target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
          for p in u.parameters():
              if p.requires_grad: p.data = p.data.float()
          _m, _unexp = u.load_state_dict({k: v.float() for k, v in S2_CK["lora_state"].items()},
                                         strict=False)
          assert not _unexp, f"unexpected LoRA keys in the checkpoint: {list(_unexp)[:4]}"
          n_lora = len(S2_CK["lora_state"])

      assert S2_CK.get("control_state"), \
          "no control_state in best.pt -- an untrained control branch cannot paint a layout"
      c.load_state_dict({k: v.float() for k, v in S2_CK["control_state"].items()})
      # The checkpoint stores control_state and lora_state as fp16 (Stage 2 halves them on save),
      # so casting the control branch back to fp16 here is lossless relative to what was actually
      # saved -- it recovers no precision that the checkpoint ever had, and roughly halves the
      # branch's memory bandwidth, which is where a T4 spends its time.
      c = c.to(CTRL_DTYPE)
      c.requires_grad_(False)
      u.eval(); c.eval()

      v  = AutoencoderKL.from_pretrained(S2_CFG["SD_MODEL_ID"], subfolder="vae").to(device).eval()
      v.requires_grad_(False)
      te = CLIPTextModel.from_pretrained(S2_CFG["SD_MODEL_ID"], subfolder="text_encoder",
                                         torch_dtype=torch.float16).to(device).eval()
      te.requires_grad_(False)
      cy = None
      if USE_CANNY:
          cy = ControlNetModel.from_pretrained(S2_CFG["CANNY_ID"],
                                               torch_dtype=torch.float16).to(device).eval()
          cy.requires_grad_(False)
      return {"device": device, "unet": u, "cn": c, "vae": v, "te": te, "canny": cy,
              "scaling": float(v.config.scaling_factor), "emb": {}, "n_lora": n_lora}

tokenizer = CLIPTokenizer.from_pretrained(S2_CFG["SD_MODEL_ID"], subfolder="tokenizer")
DDIM_CFG  = DDIMScheduler.from_pretrained(S2_CFG["SD_MODEL_ID"], subfolder="scheduler").config

ENGINES = {}
for d in DEVICES:
    t0 = time.time()
    try:
        ENGINES[d] = build_engine(d)
        print(f"  engine on {d} ready in {time.time()-t0:.0f}s | "
              f"{torch.cuda.memory_allocated(d)/1e9:.2f} GB resident")
    except Exception as e:
        # A device that will not build is dropped, not fatal. Extra GPUs are an optimisation;
        # losing one costs speed, and there is no version of this notebook where that should
        # cost you the run.
        print(f"  {d} FAILED to build ({type(e).__name__}: {e}) -- dropping it")
assert ENGINES, "no GPU could host the Stage-2 model -- this one really is fatal"
DEVICES = [d for d in DEVICES if d in ENGINES]
ENG0 = ENGINES[DEVICES[0]]

# ---- condition, canny and captions: verbatim from Stage 2 sections 9 and 10 ----
EYE7 = np.eye(cfg.N_COND_CH, dtype=np.float32)

def cond_from_index(lab):
    return np.ascontiguousarray(EYE7[to_code(lab)].transpose(2, 0, 1))

def canny_from_index(lab):
    code = to_code(lab)
    e = np.zeros(code.shape, np.uint8)
    dh = (code[:, 1:] != code[:, :-1]); dv = (code[1:, :] != code[:-1, :])
    e[:, 1:] |= dh; e[:, :-1] |= dh; e[1:, :] |= dv; e[:-1, :] |= dv
    return np.ascontiguousarray(np.repeat(e[None].astype(np.float32), 3, axis=0))

NOUN = {0: "residential rooftops", 1: "road network", 2: "river",
        3: "forest", 4: "bare unused land", 5: "agricultural fields"}
AREA_CLASSES = {0, 3, 4, 5}
THIN_CLASSES = (1, 2)
THIN_FLOOR   = 0.005

def caption_from_index(lab):
    code = to_code(lab)
    h = np.bincount(code.ravel(), minlength=cfg.N_COND_CH)[:cfg.N_CLASSES].astype(np.float64)
    s = h / max(h.sum(), 1.0)
    keep = [int(c) for c in np.argsort(-s) if s[c] >= 0.02][:3]
    for c in THIN_CLASSES:
        if s[c] >= THIN_FLOOR and c not in keep: keep.append(c)
    parts = []
    for c in sorted(keep, key=lambda c: -s[c]):
        q = ""
        if c in AREA_CLASSES:
            q = "dense " if s[c] >= 0.35 else ("scattered " if s[c] < 0.10 else "")
        parts.append(q + NOUN[c])
    return "aerial satellite view of Kathmandu Valley, " + ", ".join(parts)

@torch.no_grad()
def embed_cached(eng, caps):
    cache = eng["emb"]
    miss = sorted({c for c in caps if c not in cache})
    if miss:
        tok = tokenizer(miss, padding="max_length", max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt")
        e = eng["te"](tok.input_ids.to(eng["device"]))[0]
        for c, v in zip(miss, e): cache[c] = v
    return torch.stack([cache[c] for c in caps])

def cn_unet(eng, latents, t, emb, cond, cond_scale, extra=None):
    u, c = eng["unet"], eng["cn"]
    down, mid = c(latents.to(c.dtype), t, encoder_hidden_states=emb.to(c.dtype),
                  controlnet_cond=cond.to(c.dtype), conditioning_scale=cond_scale,
                  return_dict=False)
    down = list(down)
    for net_x, cond_x, sc_x in (extra or []):
        d2, m2 = net_x(latents.to(net_x.dtype), t, encoder_hidden_states=emb.to(net_x.dtype),
                       controlnet_cond=cond_x.to(net_x.dtype), conditioning_scale=sc_x,
                       return_dict=False)
        down = [a + b.to(a.dtype) for a, b in zip(down, d2)]
        mid = mid + m2.to(mid.dtype)
    down = [d.to(u.dtype) for d in down]
    return u(latents, t, encoder_hidden_states=emb,
             down_block_additional_residuals=down,
             mid_block_additional_residual=mid.to(u.dtype)).sample

@torch.no_grad()
def vae_decode(eng, lat):
    img = eng["vae"].decode((lat / eng["scaling"]).float()).sample
    return (img.float() / 2 + 0.5).clamp(0, 1)

@torch.no_grad()
def generate_images(labels, seed=0, steps=None, guidance=None, cond_scale=None,
                    use_canny=None, batch=None, progress=False, eng=None, seeds=None):
    # labels: list/array of (H, W) class-index masks. Returns (N, H, W, 3) uint8 RGB.
    eng        = eng or ENG0
    dev        = eng["device"]
    steps      = steps or DDIM_STEPS
    guidance   = GUIDANCE if guidance is None else guidance
    cond_scale = COND_SCALE if cond_scale is None else cond_scale
    use_canny  = USE_CANNY if use_canny is None else use_canny
    batch      = batch or cfg.IMG_BATCH
    do_cfg     = guidance > 1.0
    out = []
    # DEVICE CONTEXT. A tensor knows which GPU it lives on; a *kernel launch* does not -- it
    # goes to the calling thread's current device, which defaults to cuda:0 in every thread.
    # So a worker thread holding cuda:1 tensors would launch convolutions on cuda:0 against
    # cuda:1 addresses, and CUDA reports that as "an illegal memory access was encountered"
    # somewhere far from the real cause. The context manager sets the current device for the
    # duration and restores it afterwards, so this is safe to call from any thread.
    with torch.cuda.device(dev):
        # SIZE GUARD. ControlNet's conditioning embedding downsamples by 8, so a 512 mask gives the
        # 64x64 condition the latent expects and a 128 mask gives 16x16 -- which fails deep inside
        # the control branch with an unhelpful "tensor a (64) must match tensor b (16)". Normalise
        # here so no caller can make that mistake.
        labels = [upsample_mask(l) if l.shape[0] == cfg.GEN_SIZE else l for l in labels]
        for l in labels:
            assert l.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE), (
                f"generate_images needs {cfg.IMG_SIZE}px class-index masks, got {l.shape}")
        for s in range(0, len(labels), batch):
            chunk = list(labels[s:s + batch]); b = len(chunk)
            cond = torch.from_numpy(np.stack([cond_from_index(l) for l in chunk])).to(dev)
            emb  = embed_cached(eng, [caption_from_index(l) for l in chunk])
            neg  = embed_cached(eng, [""] * b)
            e_all = torch.cat([neg, emb]) if do_cfg else emb
            c_all = torch.cat([cond, cond]) if do_cfg else cond
            extra = []
            if use_canny and eng["canny"] is not None:
                cc = torch.from_numpy(np.stack([canny_from_index(l) for l in chunk])).to(dev)
                extra = [(eng["canny"], torch.cat([cc, cc]) if do_cfg else cc, CANNY_SCALE)]
            sch = DDIMScheduler.from_config(DDIM_CFG); sch.set_timesteps(steps, device=dev)
            # Noise is drawn PER IMAGE on the CPU from a generator seeded by that image's own key.
            # Per image, not per batch: a batch-level seed would make the latent depend on how the
            # work happened to be chunked, so a resumed session that re-chunks the leftovers would
            # paint different images at the same indices. Keying each latent to its absolute index
            # makes the output a pure function of the index -- independent of batch size, of which
            # GPU took the chunk, and of how many times the session died.
            keys = (seeds[s:s + b] if seeds is not None else [seed + s + k for k in range(b)])
            gg = torch.Generator()
            _z = []
            for kk in keys:
                gg.manual_seed(cfg.SEED + int(kk))
                _z.append(torch.randn(4, cfg.LATENT_SIZE, cfg.LATENT_SIZE, generator=gg))
            lat = torch.stack(_z).to(dev) * sch.init_noise_sigma
            for t in sch.timesteps:
                x = torch.cat([lat] * 2) if do_cfg else lat
                x = sch.scale_model_input(x, t)
                with torch.autocast("cuda", enabled=cfg.AMP):
                    eps = cn_unet(eng, x, t, e_all, c_all, cond_scale, extra)
                eps = eps.float()
                if do_cfg:
                    u_, c_ = eps.chunk(2); eps = u_ + guidance * (c_ - u_)
                lat = sch.step(eps, t, lat).prev_sample
            img = vae_decode(eng, lat).cpu().numpy().transpose(0, 2, 3, 1)
            out.append((img * 255).round().astype(np.uint8))
            if progress: print(f"    images {min(s+b, len(labels))}/{len(labels)}", flush=True)
        return np.concatenate(out)

S2_ARCH = S2_CK.get("arch", "?")
_best = next((f"{k}={S2_CK[k]}" for k in
              ("best_score", "best_metric", "best_miou", "best_val", "val_loss")
              if k in S2_CK), "n/a")
print(f"\nStage 2 loaded: {cfg.S2_REPO}")
print(f"  step {S2_CK.get('step')} | best {_best} | arch {S2_ARCH}")
print(f"  IS THIS THE FINAL CHECKPOINT? Compare step {S2_CK.get('step')} against your Stage-2 run "
      "before letting section 18 paint 4824 images.")
print(f"  LoRA tensors loaded: {ENG0['n_lora']} | control tensors: {len(S2_CK['control_state'])}")
print(f"  base {S2_CFG['SD_MODEL_ID']} + {S2_CFG['CONTROLNET_ID']}")
print("\n--- RESOLVED SAMPLING HYPER-PARAMETERS (inherited unless overridden in CFG) ---")
for nm, val, src in [("DDIM_STEPS", DDIM_STEPS, cfg.DDIM_STEPS),
                     ("GUIDANCE", GUIDANCE, cfg.GUIDANCE),
                     ("COND_SCALE", COND_SCALE, cfg.COND_SCALE),
                     ("CANNY_SCALE", CANNY_SCALE, cfg.CANNY_SCALE),
                     ("USE_CANNY", USE_CANNY, cfg.USE_CANNY)]:
    print(f"  {nm:<12} {str(val):<8} ({'CFG OVERRIDE' if src is not None else 'inherited from best.pt'})")

_t0 = time.time()
_smoke = generate_images([upsample_mask(MASKS128[cfg.R_MULTIPLIERS[0]][0])],
                         seed=0, steps=8, batch=1)
print(f"\nsmoke sample: {_smoke.shape} in {time.time()-_t0:.1f}s at 8 DDIM steps "
      "(8 steps is the smoke test only -- expect mush)")
print(f"condition {cfg.N_COND_CH}x{cfg.IMG_SIZE}x{cfg.IMG_SIZE} -> "
      f"{cfg.IMG_SIZE//8}x{cfg.IMG_SIZE//8} embedding, latent {cfg.LATENT_SIZE}x{cfg.LATENT_SIZE}"
      " -- these must agree")
print("caption for that mask:", caption_from_index(MASKS128[cfg.R_MULTIPLIERS[0]][0]))

## 13. Evaluation module — layout fidelity, KID/FID, and the memorization check

This replaces the part-1 notebooks' confusion-matrix module and extends Stage 2's. There is no
ground-truth image for a *generated* scene, so quality is measured four ways and none of them is
optional.

### Layout fidelity — the one that matters for Stage 4
Run a frozen part-1 SegFormer on each fake image and score its prediction against **the mask that
generated it**. A fake pair is only useful as training data if the image actually depicts the label
it is paired with. Reported three ways:

- `miou_gen` — fake image vs its own fake mask
- `miou_real` — real val image vs its real mask, the **ceiling**, because the segmenter is itself
  imperfect and 0.42 against a ceiling of 0.47 is a very different result from 0.42 against 0.90
- `layout_ratio = miou_gen / miou_real` — the honest reading, floored at `LAYOUT_RATIO_FLOOR`

With `USE_SECOND_SCORER` on, the same fakes are scored by the fold-1 **random-split** model as well.
Two independently trained segmenters reading the same layout out of the same fake is much stronger
evidence than one doing so; wide disagreement means the fakes are exploiting one model's quirks.

### The condition-source ablation
Stage 2's control branch was trained on **real** masks. Stage 1's masks are post-processed samples
with different boundary statistics, so they may be out of distribution for it. `COND_SOURCE_ABLATION`
paints `COND_ABL_N` **real val masks** with the same sampler and compares layout mIoU. A large gap
means the chain is losing quality at the joint, not in either stage, and that is a finding worth its
own paragraph.

### KID and FID
KID is the headline because it is unbiased at small N; FID is secondary and biased upward at these
sample sizes. Both are reported against two references: the fold-1 **train** images (which Stage 2
saw) and the fold-1 **val** images (which it did not). The val number is the honest generalization
figure. N is printed with every value.

### Memorization — the check that can invalidate everything
Three independent levels, each with a **held-out real baseline** computed the identical way:

| level | statistic | catches |
|---|---|---|
| Inception feature | min cosine distance to any train image | semantic near-copies |
| raw pixel | min L2 on 64x64 grey thumbnails | literal tile copies |
| mask | min per-pixel disagreement at 32x32 | layout recitation (Stage 1's own check) |

Baseline = the same statistic for held-out real val tiles against the train set. `ratio = fake /
real_heldout`. **At or above 1.0 the generator is no closer to the training set than a genuinely
new real tile is, which is the definition of not memorising.** Well below 1.0 and the Stage-4 result
is void. Report it either way.

In [ ]:
from scipy import linalg
from skimage.morphology import skeletonize
from transformers import SegformerForSemanticSegmentation, SegformerConfig

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# ---- frozen part-1 segmenter: INFERENCE ONLY, never fine-tuned in this notebook ----
def load_segformer(repo_id, token=None):
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location="cuda")
    m = SegformerForSemanticSegmentation(SegformerConfig.from_dict(ck["model_config"])).cuda().eval()
    m.load_state_dict(ck["model_state"])
    m.requires_grad_(False)
    return m, ck.get("cfg", {}), ck.get("metrics", {})

def to01(stack_u8):
    # (n, H, W, 3) uint8 RGB -> (n, 3, H, W) float [0,1]. Called PER BATCH, never on the whole
    # set: 804 tiles as one float32 tensor is 2.5 GB of host RAM and would OOM the kernel.
    a = np.ascontiguousarray(np.asarray(stack_u8).transpose(0, 3, 1, 2))
    return torch.from_numpy(a).float() / 255.0

@torch.no_grad()
def segment_rgb01(m, imgs_u8, batch=8):
    # imgs_u8: (N, H, W, 3) uint8 RGB. Normalisation matches part-1 preprocessing exactly.
    out = []
    for i in range(0, len(imgs_u8), batch):
        x = ((to01(imgs_u8[i:i + batch]).cuda() - IMNET_MEAN) / IMNET_STD)
        with torch.cuda.amp.autocast():
            lo = m(pixel_values=x).logits
            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
        out.append(lo.float().argmax(1).cpu())
    return torch.cat(out)

# ---- confusion matrix and metrics: copied verbatim from the part-1 notebooks ----
class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64)
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)
    user = tp / np.maximum(cm.sum(0), 1e-12)
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {"mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)), "OA": float(po),
            "FWIoU": float((freq * iou).sum()), "kappa": float((po - pe) / max(1 - pe, 1e-12)),
            "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                              "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                              "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)}}

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

def layout_scores(preds, labels, names):
    cm = ConfMat(cfg.N_CLASSES)
    for p, t in zip(preds, labels):
        cm.update(torch.as_tensor(np.asarray(p)), torch.as_tensor(np.asarray(t)))
    out = metrics_from_cm(cm.numpy(), names)
    for c, nm in ((1, "Road"), (2, "River")):
        vals = [v for p, t in zip(preds, labels)
                for v in [cldice_pair(np.asarray(p) == c, np.asarray(t) == c)] if v is not None]
        out[f"clDice_{nm}"] = float(np.mean(vals)) if vals else float("nan")
    return out

# ---- Inception features, FID and KID ----
from torchvision.models import inception_v3, Inception_V3_Weights
_INCEPTION = None
def get_inception():
    global _INCEPTION
    if _INCEPTION is None:
        m = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
        m.fc = nn.Identity()
        _INCEPTION = m.eval().cuda().requires_grad_(False)
    return _INCEPTION

@torch.no_grad()
def inception_feats(imgs_u8, batch=16):
    # imgs_u8: (N, H, W, 3) uint8 RGB, converted per batch for the same RAM reason as above
    m = get_inception(); out = []
    for i in range(0, len(imgs_u8), batch):
        x = to01(imgs_u8[i:i + batch]).cuda()
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        out.append(m((x - IMNET_MEAN) / IMNET_STD).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

def fid_from_feats(f1, f2, eps=1e-6):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    covmean, _ = linalg.sqrtm(s1.dot(s2), disp=False)
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * eps
        covmean = linalg.sqrtm((s1 + off).dot(s2 + off))
    if np.iscomplexobj(covmean): covmean = covmean.real
    d = mu1 - mu2
    return float(d.dot(d) + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))

def kid_from_feats(f1, f2, subset_size=None, subsets=None, seed=0):
    # unbiased MMD^2 with the standard polynomial kernel (degree 3, gamma 1/d, coef 1)
    m = min(subset_size or cfg.KID_SUBSET_SIZE, len(f1), len(f2))
    n_sub = subsets or cfg.KID_SUBSETS
    d = f1.shape[1]; rng = np.random.RandomState(seed); vals = []
    for _ in range(n_sub):
        x = f1[rng.choice(len(f1), m, replace=False)]
        y = f2[rng.choice(len(f2), m, replace=False)]
        kxx = (x.dot(x.T) / d + 1) ** 3
        kyy = (y.dot(y.T) / d + 1) ** 3
        kxy = (x.dot(y.T) / d + 1) ** 3
        vals.append((kxx.sum() - np.trace(kxx)) / (m * (m - 1))
                    + (kyy.sum() - np.trace(kyy)) / (m * (m - 1))
                    - 2 * kxy.mean())
    return float(np.mean(vals)), float(np.std(vals)), int(m), int(n_sub)

# ---- memorization, three levels ----
def nn_feat(qf, rf, chunk=64):
    # min cosine distance of each query feature to the whole reference set
    q = qf / np.maximum(np.linalg.norm(qf, axis=1, keepdims=True), 1e-12)
    r = rf / np.maximum(np.linalg.norm(rf, axis=1, keepdims=True), 1e-12)
    d = np.empty(len(q)); j = np.empty(len(q), dtype=int)
    for i in range(0, len(q), chunk):
        sim = q[i:i + chunk].dot(r.T)
        j[i:i + chunk] = sim.argmax(1); d[i:i + chunk] = 1.0 - sim.max(1)
    return d, j

def thumbs(stack_rgb, size=64):
    # (N, H, W, 3) uint8 -> (N, size*size) float64 grey in [0,1] units, mean-centred per image
    # (centring kills a global brightness offset, which is not evidence of copying).
    # float64 and the /255 are not cosmetic: nn_pixel expands ||x-y||^2, and in float32 at raw
    # grey magnitudes the cancellation error puts an exact self-match at distance 0.08 instead
    # of 0, which would silently flatter the memorization check.
    out = []
    for a in stack_rgb:
        g = cv2.cvtColor(np.asarray(a), cv2.COLOR_RGB2GRAY)
        g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float64) / 255.0
        out.append((g - g.mean()).ravel())
    return np.stack(out).astype(np.float64)

def nn_pixel(qt, rt, chunk=64):
    qt = np.asarray(qt, dtype=np.float64); rt = np.asarray(rt, dtype=np.float64)
    d = np.empty(len(qt)); j = np.empty(len(qt), dtype=int)
    rn = (rt ** 2).sum(1)
    for i in range(0, len(qt), chunk):
        x = qt[i:i + chunk]
        dist = np.maximum((x ** 2).sum(1)[:, None] + rn[None] - 2 * x.dot(rt.T), 0.0)
        j[i:i + chunk] = dist.argmin(1); d[i:i + chunk] = np.sqrt(dist.min(1)) / qt.shape[1] ** 0.5
    return d, j

def nn_mask(query, ref, size=32, chunk=64):
    # min per-pixel disagreement of each query mask against the whole reference set (Stage 1's check)
    def flat(s):
        return np.stack([cv2.resize(to_code(a), (size, size),
                                    interpolation=cv2.INTER_NEAREST).ravel() for a in s])
    Q, R = flat(query), flat(ref)
    d = np.empty(len(Q)); j = np.empty(len(Q), dtype=int)
    for i in range(0, len(Q), chunk):
        dd = (Q[i:i + chunk, None, :] != R[None, :, :]).mean(-1)
        j[i:i + chunk] = dd.argmin(1); d[i:i + chunk] = dd.min(1)
    return d, j

def triptych(reals, labels, gens, n=4, title=None, cell=2.6):
    n = min(n, len(gens))
    fig, axes = plt.subplots(n, 3, figsize=(3 * cell, n * cell))
    axes = np.atleast_2d(axes)
    for i in range(n):
        for j, (im, ttl) in enumerate([(reals[i], "real"), (colorize(labels[i]), "mask"),
                                       (gens[i], "generated")]):
            axes[i, j].imshow(im); axes[i, j].axis("off")
            if i == 0: axes[i, j].set_title(ttl, fontsize=10)
    if title: fig.suptitle(title, fontsize=11)
    fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    return fig

SEG_BLOCK, _sc, SEG_METRICS = load_segformer(cfg.SEG_REPO_BLOCK, token=HF_TOKEN)
print(f"frozen scorer (headline): {cfg.SEG_REPO_BLOCK}")
print(f"  published val mIoU {SEG_METRICS.get('mIoU', float('nan')):.4f} | "
      f"split {_sc.get('SPLIT_MODE')} fold {_sc.get('FOLD')}")
assert _sc.get("FOLD", cfg.FOLD) == cfg.FOLD, "the headline scorer is not a fold-1 model"

SEG_RANDOM = None
if cfg.USE_SECOND_SCORER:
    SEG_RANDOM, _sc2, _sm2 = load_segformer(cfg.SEG_REPO_RANDOM, token=HF_TOKEN)
    print(f"frozen scorer (agreement): {cfg.SEG_REPO_RANDOM} | "
          f"published val mIoU {_sm2.get('mIoU', float('nan')):.4f}")

# ---- the real reference stacks in RGB ----
REAL_TRAIN_RGB = np.stack([cv2.cvtColor(real_img(r[2]), cv2.COLOR_BGR2RGB) for r in tr_recs])
REAL_VAL_RGB   = np.stack([cv2.cvtColor(real_img(va_recs[i][2]), cv2.COLOR_BGR2RGB)
                           for i in ORIG_VAL])
REAL_VAL_LAB   = np.stack([real_lab(va_recs[i][2], va_recs[i][3]) for i in ORIG_VAL])
# REAL_TRAIN_RGB rows are in tr_recs order, which is what nn_feat / nn_pixel return indices into.
if cfg.PRELOAD_RAM:
    RAM_IMG.clear(); gc.collect()      # the two stacks above supersede it; ~950 MB back
    print("  RAM_IMG released -- the RGB reference stacks supersede it")
print(f"\nreference sets: train RGB {REAL_TRAIN_RGB.shape} | val RGB {REAL_VAL_RGB.shape} "
      f"({(REAL_TRAIN_RGB.nbytes + REAL_VAL_RGB.nbytes)/1e9:.2f} GB)")
print("metrics module ready | KID is the headline, FID secondary, N printed with both")

## 14. Efficiency — read this before you walk away

This is the cell that tells you whether the run you are about to start fits in the session you have.
It times a real batch through the full chained sampler at the real settings, then projects.

It reports **two** rates. `PER_GPU_SEC` is what one T4 does. `SEC_PER_IMG` divides that by the
number of workers, because sampling is embarrassingly parallel and section 17 runs one model
replica per visible GPU. On a `T4 x2` kernel the second figure is what the projection uses.

It also reports how many pairs are already on disk, so a resumed session projects the *remaining*
work rather than the whole ladder.

Reported: seconds per image, peak VRAM during sampling (which is where the peak lives in this
notebook — there is no optimizer state), throughput in pairs per hour, the projected wall-clock for
the whole ladder, and **how many Kaggle sessions that implies**.

If the projection exceeds `MAX_HOURS`, that is fine and expected: section 18 stops cleanly at the
budget and section 17's resume picks the run back up at the exact pair it stopped on. What you must
not do is start the run without reading this number, discover at hour 11 that you are 40% through,
and lose the session to a hard kill mid-write.

The disk projection matters too. A 512x512 RGB PNG of satellite imagery is 350-500 kB and there are
`TOTAL_PAIRS` of them plus masks plus tarballs. Kaggle gives you 20 GB of working space and the
tarball step needs room for a second copy of whatever it is compressing.

In [ ]:
def bench_chain(eng, n=None):
    n = n or cfg.IMG_BATCH
    dev = eng["device"]
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(dev)
    labs = [upsample_mask(m) for m in MASKS128[cfg.R_MULTIPLIERS[0]][:n]]   # 512, as section 17
    # Warm up at the SAME batch size as the timed run. cudnn.benchmark=True makes the first
    # forward at any new tensor shape run an exhaustive conv-algorithm search across every layer
    # of the UNet and both control branches. Warming at batch=1 and timing at batch=8 charges
    # the entire autotune bill to the stopwatch: that is why an earlier run of this cell reported
    # 117 s/image for a sampler that then produced 4084 images at 9.2 s/image.
    _ = generate_images(labs, seed=0, batch=cfg.IMG_BATCH, eng=eng)
    torch.cuda.reset_peak_memory_stats(dev)
    t0 = time.time()
    _ = generate_images(labs, seed=1, batch=cfg.IMG_BATCH, eng=eng)
    dt = time.time() - t0
    return dt / n, torch.cuda.max_memory_allocated(dev) / 1e9

# Bench EVERY device, each from its own thread, because that is how section 17 will drive them.
# Benching only cuda:0 from the main thread exercises none of the multi-device plumbing and lets
# a device-context bug survive until the ladder is already running.
_bench = {}
def _bench_one(d):
    torch.cuda.set_device(d)
    try:    _bench[d] = bench_chain(ENGINES[d])
    except Exception as e: _bench[d] = e
if len(DEVICES) == 1:
    _bench_one(DEVICES[0])                      # single device: no threads at all, ever
else:
    _th = [threading.Thread(target=_bench_one, args=(d,)) for d in DEVICES]
    for t in _th: t.start()
    for t in _th: t.join()
# A device that fails its bench is DROPPED. It is not an error and it does not stop the run:
# extra GPUs are a speed optimisation, and an optimisation must never be able to take down the
# thing it was meant to speed up. Falling back to one working GPU costs you time; raising here
# costs you the session.
_good, _bad = [], []
for d in DEVICES:
    v = _bench.get(d)
    if isinstance(v, Exception) or v is None:
        _bad.append((d, v)); continue
    _good.append(d)
    print(f"  {d}: {v[0]:.2f} s/image, peak {v[1]:.2f} GB")
for d, e in _bad:
    print(f"  {d}: DROPPED -- {type(e).__name__}: {str(e)[:160]}")
    ENGINES.pop(d, None)
if _bad and _good:
    print(f"\n  continuing on {_good}. This costs speed, nothing else: the per-image latent is")
    print("  keyed to the absolute pair index, so the output is identical either way.")
assert _good, ("every GPU failed its bench. Set cfg.MULTI_GPU = False and rerun -- the "
               "single-device path is the one that has actually run this stack end to end.")
DEVICES = _good
ENG0 = ENGINES[DEVICES[0]]
_bench = {d: _bench[d] for d in DEVICES}

PER_GPU_SEC = max(v[0] for v in _bench.values())      # the slowest GPU sets the pace
PEAK_GB     = max(v[1] for v in _bench.values())
N_WORKERS = len(DEVICES)
SEC_PER_IMG = PER_GPU_SEC / N_WORKERS            # effective, with all GPUs working in parallel
PAIRS_PER_H = 3600.0 / max(SEC_PER_IMG, 1e-9)
PROJ_H = TOTAL_PAIRS * SEC_PER_IMG / 3600.0
BYTES_PER_PAIR = 0.45e6 + 0.02e6                # empirical: 512 RGB PNG + index PNG
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
# counted by glob, not by scan_done() -- that lives in section 17, after this cell
DONE_ALREADY = sum(len(list(Path(R_DIR(r)).glob("genmask*.png")))
                   for r in cfg.R_MULTIPLIERS if Path(R_DIR(r)).is_dir())
REMAIN = TOTAL_PAIRS - DONE_ALREADY

print(f"--- chained sampler at DDIM {DDIM_STEPS}, guidance {GUIDANCE}, canny {USE_CANNY} ---")
print(f"  {PER_GPU_SEC:.2f} s / image on ONE gpu | peak VRAM {PEAK_GB:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  {N_WORKERS} worker(s) -> {SEC_PER_IMG:.2f} s / image effective | "
      f"{PAIRS_PER_H:.0f} pairs / hour")
print(f"  already on disk: {DONE_ALREADY} | remaining: {REMAIN}")
print(f"  projected for the remaining {REMAIN}: {REMAIN * SEC_PER_IMG / 3600.0:.1f} h "
      f"-> {math.ceil(REMAIN * SEC_PER_IMG / 3600.0 / cfg.MAX_HOURS)} session(s) at "
      f"{cfg.MAX_HOURS} h each")
if PEAK_GB < 0.45 * torch.cuda.get_device_properties(0).total_memory / 1e9:
    print(f"  headroom: peak is {PEAK_GB:.1f} GB -- try raising IMG_BATCH from {cfg.IMG_BATCH} "
          "to 8. Larger batches keep the T4's SMs busy and are a free speedup; the per-image "
          "latent seed is keyed to the absolute index, so batching does not change the output.")
print(f"  disk: {TOTAL_PAIRS * BYTES_PER_PAIR / 1e9:.2f} GB of pairs | {free_gb:.1f} GB free")
if TOTAL_PAIRS * BYTES_PER_PAIR * 2 / 1e9 > free_gb:
    print("  WARNING: tarring all sets in one session may fill /kaggle/working. Push and delete")
    print("  each shard as it completes (section 21 does this when PUSH_SHARDS is on).")

EFF = {"sec_per_image_1gpu": PER_GPU_SEC, "sec_per_image_effective": SEC_PER_IMG,
       "workers": N_WORKERS, "devices": DEVICES,
       "pairs_per_hour": PAIRS_PER_H, "peak_vram_gb": PEAK_GB, "projected_hours": PROJ_H,
       "ddim_steps": DDIM_STEPS, "guidance": GUIDANCE, "use_canny": USE_CANNY,
       "control_dtype": str(CTRL_DTYPE), "img_batch": cfg.IMG_BATCH,
       "s1_ddim_steps": S1_STEPS,
       "unet_params_M": sum(p.numel() for p in ENG0["unet"].parameters()) / 1e6,
       "control_params_M": sum(p.numel() for p in ENG0["cn"].parameters()) / 1e6}
print("\nnote: these are SAMPLING costs. Stage 4's training cost is a separate budget entirely --")
print("five models x three R values on a set 4x the size of fold-1 train is the expensive part.")

## 15. Hugging Face persistence

Stage 3 has no weights to push, so the repo layout differs from Stages 1 and 2. What lives here is
the **product**, not a model:

- `masks_R{r}_{tag}.npy` — the selected mask pools, so a resumed session repaints the same masks
- `fake_pairs_R{r}_{tag}.tar.gz` — one shard per R set, the Stage-4 input
- `fake_pairs_manifest.json` — provenance: both upstream repos, both checkpoint steps, the resolved
  sampling hyper-parameters, the post-processing knobs, the split SHA1, and the headline metrics
- `results_{tag}.json` / `.csv` / `history_{tag}.csv` — the numbers for the paper
- `fold1_split.json` — so a reader can reconstruct exactly which real tiles were off-limits
- `README.md` — the model card, which for this repo is a **dataset** card

The manifest is the important artefact. A fake-pair archive with no provenance is unusable a month
later: you cannot tell which mask model, which image model, which guidance scale or which fold
produced it, and therefore cannot defend the Stage-4 result. `hf_push` swallows failures and keeps
going, because losing a push should never cost you the seven hours of sampling that preceded it.

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False,
            repo_type="model")

def hf_push(local_path, name, msg, repo=None):
    try:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                        repo_id=repo or HF_REPO, token=HF_TOKEN, commit_message=msg)
        print(f"  pushed {name} ({os.path.getsize(local_path)/1e6:.1f} MB)")
        return True
    except Exception as e:
        print("  HF push failed (continuing):", type(e).__name__, e)
        return False

def hf_try_download(name, repo=None):
    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

PROVENANCE = {
    "stage": 3, "run": RUN_NAME,
    "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "train_sha1": split_sha1,
    "n_real_train_pairs": N_REAL, "r_multipliers": list(cfg.R_MULTIPLIERS),
    "r_sizes": {f"R{k}": v for k, v in R_SIZES.items()},
    "independent_draws": cfg.INDEPENDENT_DRAWS,
    "stage1": {"repo": cfg.S1_REPO, "epoch": S1_CK.get("epoch"),
               "select_on": S1_CK.get("select_on"), "best_score": S1_CK.get("best_score"),
               "arch": S1_CK.get("arch"), "ddim_steps": S1_STEPS,
               "label_coding": S1_CFG.get("LABEL_CODING"),
               "prediction_type": S1_CFG.get("PREDICTION_TYPE"),
               "zero_terminal_snr": S1_CFG.get("ZERO_TERMINAL_SNR")},
    "stage2": {"repo": cfg.S2_REPO, "step": S2_CK.get("step"), "arch": S2_ARCH,
               "ablation": S2_CFG.get("ABLATION"), "cond_mode": S2_CFG.get("COND_MODE"),
               "sd_model": S2_CFG.get("SD_MODEL_ID"), "controlnet": S2_CFG.get("CONTROLNET_ID"),
               "lora_rank": S2_CFG.get("LORA_RANK")},
    "sampling": {"ddim_steps": DDIM_STEPS, "guidance": GUIDANCE, "cond_scale": COND_SCALE,
                 "canny": USE_CANNY, "canny_scale": CANNY_SCALE,
                 "inherited": {k: getattr(cfg, k) is None for k in
                               ("DDIM_STEPS", "GUIDANCE", "COND_SCALE", "CANNY_SCALE", "USE_CANNY")},
                 "mask_seed_base": cfg.MASK_SEED_BASE, "image_seed_base": cfg.IMAGE_SEED_BASE,
                 "seed": cfg.SEED, "img_batch": cfg.IMG_BATCH,
                 "devices": DEVICES, "control_dtype": str(CTRL_DTYPE)},
    "post_processing": {"modal": PP_KNOBS[0], "close": PP_KNOBS[1],
                        "min_px_thin": PP_KNOBS[2], "min_px_bulk": PP_KNOBS[3],
                        "pool_ratio": cfg.MASK_POOL_RATIO, "filter": cfg.FILTER_MASKS},
    "scorers": {"headline": cfg.SEG_REPO_BLOCK,
                "agreement": cfg.SEG_REPO_RANDOM if cfg.USE_SECOND_SCORER else None},
    "contract": {"image": "genimage{i}.png, BGR on disk (cv2), 512x512 uint8",
                 "mask": "genmask{i}.png, single channel, values 0..5 semantic + 255 ignore",
                 "class_names": list(cfg.CLASS_NAMES)},
}
json.dump(PROVENANCE, open(f"{cfg.OUT_DIR}/provenance.json", "w"), indent=2, default=str)
for r in cfg.R_MULTIPLIERS:
    p = f"{cfg.OUT_DIR}/masks_R{r}_{tag}.npy"
    if os.path.exists(p) and not MASK_META.get(r, {}).get("restored"):
        hf_push(p, os.path.basename(p), f"mask pool R{r}")
if os.path.exists(PP_PATH): hf_push(PP_PATH, os.path.basename(PP_PATH), "post-processing knobs")
print("HF ready:", HF_REPO)
print("provenance written ->", f"{cfg.OUT_DIR}/provenance.json")

## 16. Weights & Biases

Matches the logging on the part-1 baselines and both upstream stages. `resume="allow"` with a
deterministic run id means a second session continues the same run rather than starting a rival one,
which matters here more than usual because the sampling run genuinely spans sessions.

Series this notebook has that the others do not:

- **`pairs_done`** — a monotone counter per R set. This is the resume progress bar, and it is the
  thing to look at when deciding whether to start a third session.
- **`sec_per_image`** — a rolling rate. If it climbs, something is leaking VRAM and the run will die
  on the next allocation rather than slow down gracefully.
- **`presence_Road` / `presence_River`** — per-mask presence rates of the two rare classes in each
  finished set. These are logged as summary values because they, not the pooled shares, decide
  whether Stage 4 can gain anything on the classes Part 2 exists to fix.

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config={**asdict(cfg), **{f"eff/{k}": v for k, v in EFF.items()},
                                       "resolved/ddim_steps": DDIM_STEPS,
                                       "resolved/guidance": GUIDANCE,
                                       "resolved/cond_scale": COND_SCALE,
                                       "resolved/use_canny": USE_CANNY,
                                       "resolved/pp_knobs": list(PP_KNOBS),
                                       "n_real_train_pairs": N_REAL,
                                       "train_sha1": split_sha1})
    wandb.summary["stage1_repo"] = cfg.S1_REPO
    wandb.summary["stage2_repo"] = cfg.S2_REPO
    for r in cfg.R_MULTIPLIERS:
        pr = presence_rate(MASKS128[r])
        wandb.summary[f"R{r}/presence_Road"]  = float(pr[1])
        wandb.summary[f"R{r}/presence_River"] = float(pr[2])
        wandb.summary[f"R{r}/mask_KL"] = kl_div(P_REAL, class_shares(MASKS128[r]))
    print("wandb:", wandb.run.url)
else:
    print("continuing without wandb")

## 17. The pair writer

One function, `produce_set(r)`. Everything about parallelism and resumability lives in it.

**With one device it does not thread at all.** `worker()` is called directly in the main thread —
no queue handoff between threads, no thread-local CUDA device state, nothing that can go wrong in a
way that only shows up after ten minutes of model loading. This is the default and it is the path
that has actually run this stack.

**With more than one device** it builds a full model replica per GPU and starts one thread each,
pulling chunks off a shared `queue.Queue`. Not `DataParallel`, not DDP — sampling has no gradients
and nothing to synchronise. This path is a speed optimisation and it is off by default: it failed
twice on Kaggle's `T4 x2` with a CUDA illegal memory access that survived both
`torch.cuda.set_device()` in the worker and a `torch.cuda.device()` context around the sampler.
Section 14 benches every device before the ladder starts and drops any that fails, so turning it on
can cost you fifteen minutes but never a session.

**The output does not depend on how the work was divided.** Each latent is drawn from a CPU
generator seeded by `IMAGE_SEED_BASE + 1000*R + i` — the absolute pair index and nothing else. Not
the batch, not the chunk, not the device. This is the property that makes a resumed set
byte-identical to an uninterrupted one, and it is why a batch-level seed would have been wrong: a
resumed session re-chunks whatever is left over, so batch-keyed noise would repaint different
images at the same indices.

**Write-through.** Each pair is written to `PAIR_DIR/R{r}/genimage{i}.png` +
`genmask{i}.png` the moment its batch decodes. Nothing is held in memory waiting for a
final flush, so a session that is killed at pair 3100 leaves 3100 valid pairs on disk.

**Resume by scan, not by bookkeeping.** `missing_indices(r)` returns every index whose pair is not
complete on disk, and that is the work list. It does not trust a progress file, because a progress
file and a directory can disagree and the directory is the ground truth. Note this is a *set*, not
a prefix: two workers finish out of order, so the done indices are not an interval and the
contiguous-prefix scan a single-GPU version could get away with would silently repaint work that
already exists.

**Write order is mask first, image second.** A crash between the two writes then leaves a pair that
`missing_indices` classifies as incomplete and redoes, rather than an orphaned image that a naive
file count would score as finished.

**Finished pairs leave the session while it is still running.** Every `CHECKPOINT_EVERY`
pairs, `push_chunk` tars just those pairs and uploads them as
`chunk_R{r}_{lo}-{hi}_{tag}.tar.gz`. Chunks hold disjoint index sets under the same
`fake_pairs_R{r}/` arcname, so the restore above merges any number of them without conflict.
Incremental rather than whole-set: re-tarring 2412 pairs at every checkpoint would cost more
than the painting does.

This is the difference between losing a session's work and losing 250 pairs. It is also what
makes the run survive an account change, a quota reset, or a kernel that dies without
committing — none of which the Kaggle-side output mechanism can be relied on for. The flush
runs on **every** exit path: normal completion, budget stop, and worker fault alike.

At the end of each set `produce_set` prints how many pairs are on disk versus on the Hub, and
warns loudly if those numbers disagree.

**The budget check runs between chunks**, not between sets. Stopping mid-set is fine and expected;
stopping mid-chunk is not, because a partially decoded batch has nothing to write. When the budget
runs out both workers drain and return, and section 18 tars whatever exists and pushes it.

**A worker exception propagates.** If one GPU faults, the other stops and `produce_set` re-raises
in the main thread rather than returning a quietly truncated set.

The contract asserts at the end are deliberately paranoid, and they are the same asserts Stage 1 and
Stage 2 ran on their own outputs. A silent channel-order flip here would not surface until Stage 4
produced an inexplicably worse model a week from now.

In [ ]:
def _pair_paths(r, i):
    d = R_DIR(r)
    return f"{d}/genimage{i}.png", f"{d}/genmask{i}.png"

def pair_exists(r, i):
    ip, mp = _pair_paths(r, i)
    return os.path.exists(ip) and os.path.exists(mp)

def missing_indices(r):
    # Every index whose pair is not COMPLETE on disk. A torn pair (image written, mask not)
    # counts as missing and gets redone. This replaces the contiguous-prefix scan of the
    # single-GPU version: two workers finish out of order, so the set of done indices is not
    # an interval and a prefix scan would silently re-paint work that already exists.
    return [i for i in range(R_SIZES[r]) if not pair_exists(r, i)]

def scan_done(r):
    if not Path(R_DIR(r)).is_dir(): return 0
    return R_SIZES[r] - len(missing_indices(r))

def write_pair(r, i, rgb, lab):
    ip, mp = _pair_paths(r, i)
    # mask FIRST, image second: a crash between the two then leaves an incomplete pair that
    # missing_indices() will redo, rather than a mask-less image that looks done to a naive scan
    cv2.imwrite(mp, lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    # BGR on disk, matching how the part-1 loaders read images with cv2.IMREAD_COLOR
    cv2.imwrite(ip, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_PNG_COMPRESSION, 3])

# upsample_mask lives in section 10 -- sections 12 and 14 need it too

_WLOCK = threading.Lock()
_PUSHED = {r: set() for r in cfg.R_MULTIPLIERS}     # indices known to be on the Hub

def push_chunk(r, idxs):
    # Tar just these pairs and push them. Small, fast, and additive: each chunk holds a
    # disjoint set of indices under fake_pairs_R{r}/, so the restore in this same section
    # merges any number of them without conflict. Incremental rather than whole-set, because
    # re-tarring 2412 pairs at every checkpoint would cost more than the painting does.
    idxs = sorted(set(idxs) - _PUSHED[r])
    if not idxs: return True
    lo, hi = idxs[0], idxs[-1]
    name = f"chunk_R{r}_{lo:05d}-{hi:05d}_{tag}.tar.gz"
    tarp = f"{cfg.OUT_DIR}/{name}"
    if os.path.exists(tarp): os.remove(tarp)
    try:
        with tarfile.open(tarp, "w:gz") as tf:
            for i in idxs:
                ip, mp = _pair_paths(r, i)
                tf.add(mp, arcname=f"fake_pairs_R{r}/genmask{i}.png")
                tf.add(ip, arcname=f"fake_pairs_R{r}/genimage{i}.png")
        mb = os.path.getsize(tarp) / 1e6
        ok = hf_push(tarp, name, f"R{r} pairs {lo}-{hi} ({len(idxs)})")
        if ok:
            _PUSHED[r] |= set(idxs)
            print(f"    checkpoint: R{r} pairs {lo}-{hi} safe on the Hub ({mb:.0f} MB)",
                  flush=True)
        else:
            print(f"    checkpoint push FAILED for R{r} {lo}-{hi} -- pairs are still on "
                  "local disk, and the next checkpoint will retry them", flush=True)
        return ok
    except Exception as e:
        print(f"    checkpoint push errored ({type(e).__name__}: {e}) -- continuing", flush=True)
        return False
    finally:
        if os.path.exists(tarp): os.remove(tarp)

def produce_set(r, log=True):
    os.makedirs(R_DIR(r), exist_ok=True)
    target = R_SIZES[r]
    todo = missing_indices(r)
    # Anything already on disk at entry came from a restore, so it is already on the Hub.
    _PUSHED[r] |= set(range(target)) - set(todo)
    if not todo:
        print(f"R{r}: already complete ({target}/{target})")
        return target, True
    img_seed = cfg.IMAGE_SEED_BASE + 1000 * r
    print(f"\n=== R{r}: {len(todo)} pairs to paint (of {target}) on {len(DEVICES)} GPU(s) "
          f"| seed {img_seed}, DDIM {DDIM_STEPS}, guidance {GUIDANCE} ===", flush=True)

    # Contiguous chunks so each worker's generate_images call batches indices that sit together;
    # the latent seed is keyed to the absolute index, so which worker takes which chunk has no
    # effect whatsoever on the output.
    chunks = [todo[i:i + cfg.IMG_BATCH] for i in range(0, len(todo), cfg.IMG_BATCH)]
    q = queue.Queue()
    for c in chunks: q.put(c)

    state = {"done": 0, "t0": time.time(), "stop": False, "err": None, "pending": []}

    def worker(dev):
        # EVERYTHING in this thread is inside the guard, including the setup. An exception
        # raised before the work loop -- a bad device string, a missing engine -- must land in
        # state["err"] like any other, or produce_set returns a quietly empty set and section 18
        # reports "0 pairs" with no reason attached.
        try:
            # Pin this thread to its GPU. PyTorch's current device is THREAD-LOCAL and defaults
            # to cuda:0 in every new thread, so without this the cuda:1 worker launches its
            # kernels on cuda:0 against cuda:1 memory, and CUDA reports that as "an illegal
            # memory access was encountered" from whichever conv happened to be running.
            # generate_images sets the context too; this is the belt to that pair of braces.
            torch.cuda.set_device(dev)
            eng = ENGINES[dev]
            _work(dev, eng)
        except Exception as e:
            with _WLOCK:
                state["err"] = e; state["stop"] = True

    def _work(dev, eng):
        while not state["stop"]:
            try: idxs = q.get_nowait()
            except queue.Empty: return
            labs = [upsample_mask(MASKS128[r][i]) for i in idxs]
            # explicit per-image seeds keyed to the ABSOLUTE pair index, so pair i is the
            # same image whether it was painted first or last, on GPU 0 or GPU 1, in one
            # session or in four
            rgb = generate_images(labs, seeds=[img_seed + i for i in idxs],
                                  batch=len(labs), eng=eng)
            for k, i in enumerate(idxs):
                write_pair(r, i, rgb[k], labs[k])
            with _WLOCK:
                state["pending"] += list(idxs)
                if cfg.CHECKPOINT_EVERY and len(state["pending"]) >= cfg.CHECKPOINT_EVERY:
                    _batch, state["pending"] = state["pending"], []
                    push_chunk(r, _batch)
                state["done"] += len(idxs)
                d, el = state["done"], time.time() - state["t0"]
                if log and (d % max(cfg.WRITE_EVERY, cfg.IMG_BATCH)) < cfg.IMG_BATCH:
                    rate = el / max(d, 1)
                    print(f"  {d}/{len(todo)} this session | {rate:.2f} s/img effective | "
                          f"eta {(len(todo)-d)*rate/3600:.2f} h | "
                          f"budget left {budget_left():.2f} h", flush=True)
                    if WANDB:
                        wandb.log({f"R{r}/pairs_done": target - len(todo) + d,
                                   "sec_per_image": rate, "budget_left_h": budget_left()})
                if budget_left() <= 0.15:
                    state["stop"] = True
                    print(f"  session budget reached -- stopping cleanly after {d} pairs. "
                          "Save Version and rerun; produce_set() resumes from what is on disk.")

    if len(DEVICES) == 1:
        # SINGLE DEVICE: no threads, no queue handoff, no thread-local device state. Plain
        # sequential code in the main thread. This is the path that has actually run, and it is
        # the default. Do not "simplify" it into the threaded branch below.
        worker(DEVICES[0])
    else:
        threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in DEVICES]
        for t in threads: t.start()
        for t in threads: t.join()

    # Flush whatever has not been checkpointed yet, on EVERY exit path -- normal completion,
    # budget stop, or worker fault. This is the last chance these pairs get to leave the
    # session alive.
    if cfg.CHECKPOINT_EVERY and state["pending"]:
        print(f"  flushing final {len(state['pending'])} pairs of R{r} to the Hub ...",
              flush=True)
        push_chunk(r, state["pending"]); state["pending"] = []

    if state["err"] is not None:
        print(f"  {scan_done(r)}/{target} pairs on disk, {len(_PUSHED[r])} of them on the Hub; "
              "the next run resumes from there")
        raise state["err"]

    n = scan_done(r)
    ok = (n >= target)
    print(f"R{r}: {n}/{target} on disk, {len(_PUSHED[r])} on the Hub, after "
          f"{(time.time()-state['t0'])/60:.1f} min this session")
    if n > len(_PUSHED[r]):
        print(f"  WARNING: {n - len(_PUSHED[r])} pairs are NOT on the Hub. They will be lost "
              "when this session ends. Check the push errors above.")
    return n, ok

def contract_check(r):
    n = scan_done(r)
    ip, mp = _pair_paths(r, 0)
    ci = cv2.imread(ip, cv2.IMREAD_UNCHANGED)
    cm_ = cv2.imread(mp, cv2.IMREAD_UNCHANGED)
    assert ci is not None and ci.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE, 3) and ci.dtype == np.uint8, \
        f"{ip}: {None if ci is None else (ci.shape, ci.dtype)}"
    assert cm_ is not None and cm_.ndim == 2 and cm_.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE) \
        and cm_.dtype == np.uint8, f"{mp}: {None if cm_ is None else (cm_.shape, cm_.dtype)}"
    seen = set()
    for k in range(0, R_SIZES[r], max(1, R_SIZES[r] // 40)):
        seen |= set(np.unique(cv2.imread(_pair_paths(r, k)[1], cv2.IMREAD_UNCHANGED)).tolist())
    assert seen <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_INDEX}, \
        f"illegal mask values in R{r}: {sorted(seen)}"
    assert n == R_SIZES[r], f"R{r} has {n} pairs, expected {R_SIZES[r]}"
    # the mask on disk must still be the mask that generated the image
    _k = R_SIZES[r] // 2
    assert (cv2.imread(_pair_paths(r, _k)[1], cv2.IMREAD_UNCHANGED)
            == upsample_mask(MASKS128[r][_k])).all(), \
        f"R{r} pair {_k}: the mask on disk does not match MASKS128 -- the pool changed under you"
    print(f"R{r} contract OK | {n} pairs | image {ci.shape} {ci.dtype} | "
          f"mask {cm_.shape} {cm_.dtype} | mask values {sorted(seen)} | pair {_k} mask verified")
    return n

# ---- restore pairs painted in earlier sessions ------------------------------------------
# /kaggle/working does not survive a session. Anything already painted therefore has to come
# back from either an attached dataset (a previous version's Output) or the Hub, or it gets
# repainted at ~9 s each. Both sources are merged; existing local files always win.
_RPAT = re.compile(r"(?:fake_pairs_)?R(\d+)$")

def _ingest_dir(d):
    got = 0
    for sub in sorted(Path(d).rglob("*")):
        if not sub.is_dir(): continue
        m = _RPAT.match(sub.name)
        if not m: continue
        r = int(m.group(1))
        if r not in R_SIZES: continue
        os.makedirs(R_DIR(r), exist_ok=True)
        for i in missing_indices(r):
            si, sm = f"{sub}/genimage{i}.png", f"{sub}/genmask{i}.png"
            if os.path.exists(si) and os.path.exists(sm):
                di, dm = _pair_paths(r, i)
                shutil.copy(sm, dm); shutil.copy(si, di); got += 1
    return got

def _ingest_tarballs(paths):
    got = 0
    for t in paths:
        stage = f"{cfg.OUT_DIR}/_stage"
        shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
        try:
            with tarfile.open(t, "r:gz") as tf: tf.extractall(stage)
            got += _ingest_dir(stage)
        except Exception as e:
            print(f"  could not read {os.path.basename(t)}: {type(e).__name__}: {e}")
        shutil.rmtree(stage, ignore_errors=True)
    return got

_restored = 0
if cfg.PAIR_RESTORE_DIR and os.path.isdir(cfg.PAIR_RESTORE_DIR):
    _restored += _ingest_dir(cfg.PAIR_RESTORE_DIR)
    _restored += _ingest_tarballs(sorted(Path(cfg.PAIR_RESTORE_DIR).rglob("*R*.tar.gz")))
    print(f"restored {_restored} pairs from {cfg.PAIR_RESTORE_DIR}")

if cfg.RESTORE_PAIRS_FROM_HF and any(missing_indices(r) for r in cfg.R_MULTIPLIERS):
    try:
        _files = [f for f in api.list_repo_files(HF_REPO, token=HF_TOKEN)
                  if re.search(r"R\d+.*\.tar\.gz$", f)]
        if _files:
            print(f"pulling {len(_files)} pair tarball(s) from {HF_REPO}: {_files}")
            _dl = [hf_hub_download(HF_REPO, f, token=HF_TOKEN) for f in _files]
            _n = _ingest_tarballs(_dl)
            _restored += _n
            print(f"  restored {_n} pairs from the Hub")
    except Exception as e:
        print("  Hub pair restore skipped:", type(e).__name__, e)

print(f"\npair writer ready | {len(DEVICES)} worker(s) | output root {cfg.PAIR_DIR}")
for r in cfg.R_MULTIPLIERS:
    print(f"  R{r}: {scan_done(r)}/{R_SIZES[r]} pairs on disk"
          f"{' (RESTORED)' if _restored else ''}")
_todo_total = sum(len(missing_indices(r)) for r in cfg.R_MULTIPLIERS)
print(f"  {_todo_total} pairs left to paint "
      f"(~{_todo_total * PER_GPU_SEC / max(len(DEVICES),1) / 3600:.1f} h at the benched rate, "
      "though the bench runs cold and tends to over-estimate)")

## 18. Main run — the ladder, with automatic resume

Smallest set first. That is deliberate: if the session dies, you would rather own a complete R1 than
40% of an R3, because a complete R1 is already a runnable Stage-4 experiment and 40% of an R3 is
nothing.

The loop stops when `budget_left()` runs out and reports exactly where. **Save Version, start a new
session, and rerun the notebook top to bottom** — sections 6 to 12 are all cache-or-download, the
mask pools reload from `.npy`, and section 17's scan puts you back on the pair you stopped at.

`ALL_DONE` gates everything downstream. If a set is incomplete, evaluation is skipped rather than
run on a partial set, because a KID computed on 40% of R3 is not a number you want appearing in a
notebook you will read again in three weeks and half-remember.

**`PUSH_PARTIAL` is the insurance.** When the ladder is incomplete, section 18 tars each partial set
and pushes it to the Hub before the notebook ends. Section 21's hand-off package only builds for a
complete ladder, so without this a killed session would take its finished pairs with it. To resume
from a partial tarball in a fresh session, untar it into `PAIR_DIR/R{r}/` before section 18 runs;
`missing_indices` then paints only what is actually absent.

In [ ]:
STATUS = {}
for r in sorted(cfg.R_MULTIPLIERS):          # smallest first
    n, done = produce_set(r)
    STATUS[r] = {"n": n, "complete": done}
    if done:
        contract_check(r)
        if WANDB: wandb.summary[f"R{r}/complete"] = True
    else:
        break

ALL_DONE = all(STATUS.get(r, {}).get("complete") for r in cfg.R_MULTIPLIERS)
print("\n--- ladder status ---")
for r in cfg.R_MULTIPLIERS:
    s = STATUS.get(r, {"n": scan_done(r), "complete": scan_done(r) >= R_SIZES[r]})
    print(f"  R{r}: {s['n']}/{R_SIZES[r]} {'COMPLETE' if s['complete'] else 'INCOMPLETE'}")
print(f"elapsed {hours():.2f} h of {cfg.MAX_HOURS} h budget")

# ---- partial insurance: never let a killed session take finished pairs with it ----
_unpushed = {r: scan_done(r) - len(_PUSHED[r]) for r in cfg.R_MULTIPLIERS}
if cfg.CHECKPOINT_EVERY and not any(_unpushed.values()):
    print("\nevery painted pair is already on the Hub via checkpoint chunks -- "
          "no whole-set tarball needed")
elif not ALL_DONE and cfg.PUSH_PARTIAL:
    # EVERY set with pairs in it, complete or not. A finished R1 sitting in an unfinished ladder
    # is the most valuable thing on the disk and the easiest to lose: section 21 only builds the
    # hand-off package for a complete ladder, so without this a finished set would go down with
    # the session. (This cell used to skip complete sets. That was exactly backwards.)
    for r in cfg.R_MULTIPLIERS:
        n = scan_done(r)
        if n == 0: continue
        done_tag = "complete" if n == R_SIZES[r] else f"{n}of{R_SIZES[r]}"
        tarp = f"{cfg.OUT_DIR}/pairs_R{r}_{done_tag}_{tag}.tar.gz"
        if os.path.exists(tarp): os.remove(tarp)
        with tarfile.open(tarp, "w:gz") as tf:
            tf.add(R_DIR(r), arcname=f"fake_pairs_R{r}")
        print(f"R{r}: {n}/{R_SIZES[r]} pairs, {os.path.getsize(tarp)/1e6:.0f} MB")
        if hf_push(tarp, os.path.basename(tarp), f"R{r} ({n}/{R_SIZES[r]} pairs)"):
            os.remove(tarp)          # free the working disk before tarring the next set

if not ALL_DONE:
    print("\nNOT FINISHED. Save Version, start a new session, rerun top to bottom.")
    print("Sections 6-12 are cache-or-download; produce_set() repaints only the missing indices.")
    print("To resume from a partial tarball, untar it into", cfg.PAIR_DIR, "before section 18.")
else:
    print("\nall sets complete -- evaluation follows")

## 19. Evaluation of the produced sets

Runs only when the whole ladder is on disk. For each R set, on a random `EVAL_N` subsample:

1. **Layout fidelity** with the fold-1 block scorer, fake image vs its own fake mask, against the
   real-image ceiling computed on the fold-1 val tiles. Plus per-class IoU and clDice for Road and
   River, which are the classes this whole project exists to fix.
2. **Inter-model agreement**: the same fakes through the fold-1 random-split scorer, and the
   pixel-agreement rate between the two scorers' predictions on the same fake. Low agreement means
   the fakes are ambiguous, not that either model is wrong.
3. **KID and FID** against both the train reference (which Stage 2 saw) and the val reference (which
   it did not). N is printed with every value because both metrics are meaningless without it.
4. **Memorization** at all three levels with the held-out real baseline, as set out in section 13.
5. **Condition-source ablation**, once: the same sampler on `COND_ABL_N` real val masks. If layout
   mIoU there is much higher than on Stage-1 masks, the chain is losing quality at the joint.

The three R sets should be *statistically indistinguishable* on every metric here — they are
independent draws from the same generator, so a real difference would mean one pool got unlucky and
the Stage-4 curve would be confounded. Differences bigger than the KID standard deviation are worth
investigating before you run Stage 4.

In [ ]:
RESULTS = {}

if ALL_DONE:
    def load_set(r, idx):
        imgs = np.stack([cv2.cvtColor(cv2.imread(_pair_paths(r, i)[0], cv2.IMREAD_COLOR),
                                      cv2.COLOR_BGR2RGB) for i in idx])
        labs = np.stack([cv2.imread(_pair_paths(r, i)[1], cv2.IMREAD_UNCHANGED) for i in idx])
        return imgs, labs

    # ---- the ceiling and the references, computed once ----
    print("--- real-image ceiling (fold-1 val tiles, block scorer) ---")
    pred_real = segment_rgb01(SEG_BLOCK, REAL_VAL_RGB, batch=cfg.EVAL_BATCH).numpy()
    CEIL = layout_scores(pred_real, REAL_VAL_LAB, list(cfg.CLASS_NAMES))
    print(f"  mIoU {CEIL['mIoU']:.4f} | mF1 {CEIL['mF1']:.4f} | OA {CEIL['OA']:.4f} | "
          f"clDice Road {CEIL['clDice_Road']:.4f} River {CEIL['clDice_River']:.4f}  "
          f"(N={len(REAL_VAL_RGB)})")

    F_TRAIN = inception_feats(REAL_TRAIN_RGB)
    F_VAL   = inception_feats(REAL_VAL_RGB)
    T_TRAIN = thumbs(REAL_TRAIN_RGB)
    print(f"  Inception features: train {F_TRAIN.shape} | val {F_VAL.shape}")

    # ---- memorization baseline: held-out REAL val vs the train set ----
    d_feat_base, _ = nn_feat(F_VAL, F_TRAIN)
    d_pix_base, _  = nn_pixel(thumbs(REAL_VAL_RGB), T_TRAIN)
    d_mask_base, _ = nn_mask(VAL_128, TRAIN_128)
    BASE = {"feat": float(d_feat_base.mean()), "pixel": float(d_pix_base.mean()),
            "mask": float(d_mask_base.mean())}
    print(f"  memorization baseline (held-out real val vs train): "
          f"feat {BASE['feat']:.4f} | pixel {BASE['pixel']:.4f} | mask {BASE['mask']:.4f}")

    # ---- condition-source ablation, once ----
    COND_ABL = {}
    if cfg.COND_SOURCE_ABLATION:
        m = min(cfg.COND_ABL_N, len(REAL_VAL_LAB))
        print(f"\n--- condition-source ablation: painting {m} REAL val masks ---")
        t0 = time.time()
        abl_rgb = generate_images(list(REAL_VAL_LAB[:m]), seed=cfg.IMAGE_SEED_BASE + 99,
                                  batch=cfg.IMG_BATCH)
        p = segment_rgb01(SEG_BLOCK, abl_rgb, batch=cfg.EVAL_BATCH).numpy()
        s = layout_scores(p, REAL_VAL_LAB[:m], list(cfg.CLASS_NAMES))
        COND_ABL = {"n": m, "miou_on_real_masks": s["mIoU"],
                    "clDice_Road": s["clDice_Road"], "clDice_River": s["clDice_River"],
                    "minutes": (time.time() - t0) / 60}
        print(f"  layout mIoU on REAL masks {s['mIoU']:.4f} "
              f"(ratio to ceiling {s['mIoU']/max(CEIL['mIoU'],1e-9):.3f}) "
              f"in {COND_ABL['minutes']:.1f} min")
        ABL_RGB, ABL_LAB = abl_rgb, REAL_VAL_LAB[:m]

    # ---- per-R evaluation ----
    for r in cfg.R_MULTIPLIERS:
        n = R_SIZES[r]
        idx = np.random.RandomState(cfg.SEED + r).choice(n, min(cfg.EVAL_N, n), replace=False)
        idx = np.sort(idx)
        G_RGB, G_LAB = load_set(r, idx)
        G_128 = MASKS128[r][idx]
        print(f"\n=== R{r} ({n} pairs, scored on {len(idx)}) ===")

        # 1. layout fidelity
        pg = segment_rgb01(SEG_BLOCK, G_RGB, batch=cfg.EVAL_BATCH).numpy()
        SC = layout_scores(pg, G_LAB, list(cfg.CLASS_NAMES))
        ratio = SC["mIoU"] / max(CEIL["mIoU"], 1e-9)
        print(f"  layout mIoU {SC['mIoU']:.4f} | ceiling {CEIL['mIoU']:.4f} | "
              f"ratio {ratio:.3f} {'PASS' if ratio >= cfg.LAYOUT_RATIO_FLOOR else 'BELOW FLOOR'}")
        print("  per-class IoU:", {k: round(v["IoU"], 3) for k, v in SC["per_class"].items()})
        print(f"  clDice Road {SC['clDice_Road']:.4f} | River {SC['clDice_River']:.4f}")

        # 2. inter-model agreement
        AGREE = {}
        if SEG_RANDOM is not None:
            pg2 = segment_rgb01(SEG_RANDOM, G_RGB, batch=cfg.EVAL_BATCH).numpy()
            SC2 = layout_scores(pg2, G_LAB, list(cfg.CLASS_NAMES))
            agree = float((pg == pg2).mean())
            AGREE = {"miou_scorer2": SC2["mIoU"], "pixel_agreement": agree}
            print(f"  second scorer mIoU {SC2['mIoU']:.4f} | pixel agreement between "
                  f"scorers {agree:.4f}")

        # 3. KID / FID
        FG = inception_feats(G_RGB)
        kid_tr = kid_from_feats(FG, F_TRAIN); kid_va = kid_from_feats(FG, F_VAL)
        fid_tr = fid_from_feats(FG, F_TRAIN); fid_va = fid_from_feats(FG, F_VAL)
        print(f"  KID vs train {kid_tr[0]:.5f} +/- {kid_tr[1]:.5f} "
              f"(m={kid_tr[2]}, {kid_tr[3]} subsets, N_gen={len(FG)}, N_real={len(F_TRAIN)})")
        print(f"  KID vs val   {kid_va[0]:.5f} +/- {kid_va[1]:.5f}   <-- the honest one")
        print(f"  FID vs train {fid_tr:.2f} | FID vs val {fid_va:.2f}")

        # 4. memorization
        k = min(cfg.NN_CHECK_N, len(G_RGB))
        d_feat, j_feat = nn_feat(FG[:k], F_TRAIN)
        d_pix,  j_pix  = nn_pixel(thumbs(G_RGB[:k]), T_TRAIN)
        d_msk,  j_msk  = nn_mask(G_128[:k], TRAIN_128)
        MEM = {"n": k,
               "feat_mean": float(d_feat.mean()), "feat_min": float(d_feat.min()),
               "feat_ratio": float(d_feat.mean() / BASE["feat"]),
               "pixel_mean": float(d_pix.mean()), "pixel_min": float(d_pix.min()),
               "pixel_ratio": float(d_pix.mean() / BASE["pixel"]),
               "mask_mean": float(d_msk.mean()), "mask_min": float(d_msk.min()),
               "mask_ratio": float(d_msk.mean() / BASE["mask"]),
               "baseline": BASE}
        print(f"  memorization (fake vs train / held-out-real vs train):")
        for lv in ("feat", "pixel", "mask"):
            print(f"    {lv:<6} mean {MEM[f'{lv}_mean']:.4f}  min {MEM[f'{lv}_min']:.4f}  "
                  f"baseline {BASE[lv]:.4f}  ratio {MEM[f'{lv}_ratio']:.3f}"
                  f"{'  <-- SUSPICIOUS' if MEM[f'{lv}_ratio'] < 0.7 else ''}")

        # 5. mask-space realism, for the paper table
        LS_G, SH_G = landscape_table(G_128, f"R{r}")
        pr = presence_rate(G_128)
        RESULTS[f"R{r}"] = {
            "n_pairs": n, "n_scored": len(idx),
            "layout": {**{kk: SC[kk] for kk in ("mIoU", "mF1", "OA", "FWIoU", "kappa",
                                                "clDice_Road", "clDice_River")},
                       "per_class": SC["per_class"],
                       "miou_ceiling": CEIL["mIoU"], "layout_ratio": ratio,
                       **AGREE},
            "KID_train": {"mean": kid_tr[0], "std": kid_tr[1], "m": kid_tr[2],
                          "subsets": kid_tr[3], "n_gen": len(FG), "n_real": len(F_TRAIN)},
            "KID_val":   {"mean": kid_va[0], "std": kid_va[1], "m": kid_va[2],
                          "subsets": kid_va[3], "n_gen": len(FG), "n_real": len(F_VAL)},
            "FID_train": fid_tr, "FID_val": fid_va,
            "memorization": MEM,
            "mask_class_share": class_shares(MASKS128[r]).tolist(),
            "mask_KL_vs_real": kl_div(P_REAL, class_shares(MASKS128[r])),
            "mask_presence_rate": pr.tolist(),
            "mask_SHDI": float(SH_G.mean()),
            "mask_meta": MASK_META.get(r, {}),
        }
        if WANDB:
            wandb.summary[f"R{r}/layout_mIoU"] = SC["mIoU"]
            wandb.summary[f"R{r}/layout_ratio"] = ratio
            wandb.summary[f"R{r}/KID_val"] = kid_va[0]
            wandb.summary[f"R{r}/FID_val"] = fid_va
            wandb.summary[f"R{r}/mem_feat_ratio"] = MEM["feat_ratio"]
        del G_RGB, G_LAB, FG; gc.collect()

    # ---- the cross-R summary table ----
    rows = []
    for r in cfg.R_MULTIPLIERS:
        R_ = RESULTS[f"R{r}"]
        rows.append({"set": f"R{r}", "pairs": R_["n_pairs"],
                     "layout_mIoU": round(R_["layout"]["mIoU"], 4),
                     "ratio": round(R_["layout"]["layout_ratio"], 3),
                     "IoU_Road": round(R_["layout"]["per_class"]["Road"]["IoU"], 3),
                     "IoU_River": round(R_["layout"]["per_class"]["River"]["IoU"], 3),
                     "KID_val": round(R_["KID_val"]["mean"], 5),
                     "FID_val": round(R_["FID_val"], 1),
                     "mem_feat": round(R_["memorization"]["feat_ratio"], 3),
                     "mem_pix": round(R_["memorization"]["pixel_ratio"], 3),
                     "mem_mask": round(R_["memorization"]["mask_ratio"], 3),
                     "River_in%": round(100 * R_["mask_presence_rate"][2], 1)})
    SUMMARY = pd.DataFrame(rows)
    print("\n=== STAGE 3 SUMMARY ===")
    print(SUMMARY.to_string(index=False))
    print(f"\nreal-image ceiling mIoU {CEIL['mIoU']:.4f} | layout ratio floor "
          f"{cfg.LAYOUT_RATIO_FLOOR}")
    if cfg.COND_SOURCE_ABLATION:
        print(f"condition-source ablation: mIoU on REAL masks "
              f"{COND_ABL['miou_on_real_masks']:.4f} vs on Stage-1 masks "
              f"{RESULTS[f'R{cfg.R_MULTIPLIERS[0]}']['layout']['mIoU']:.4f}")
    print("\nread the three memorization ratios first. At or above ~1.0 the fakes are no closer to")
    print("the training set than a genuinely new real tile is. Well below 0.7 and Stage 4's result")
    print("would be a memorization artefact -- report it either way.")
else:
    print("ladder incomplete -- evaluation skipped. Finish section 18 first.")

## 20. Persist results

Writes `results_{tag}.json` (full, including per-class breakdowns and the whole memorization block),
a flat `.csv` with one row per R set for the cross-stage comparison table, and
`ladder_{tag}.csv` for the figures. All three go to the Hub alongside `provenance.json` and
`fold1_split.json`, so the numbers survive the session and can be traced to the exact pair of
checkpoints that produced them.

The flat CSV is the row that goes into the paper's Stage-3 table. Keep the column names stable
across stages; the cross-stage table is assembled by concatenating these files.

In [ ]:
if ALL_DONE:
    final = {"stage": 3, "run": RUN_NAME, "tag": tag,
             "provenance": PROVENANCE, "efficiency": EFF,
             "ceiling": CEIL, "memorization_baseline": BASE,
             "condition_source_ablation": COND_ABL,
             "real_class_share": P_REAL.tolist(),
             "real_presence_rate": presence_rate(TRAIN_128).tolist(),
             "sets": RESULTS,
             "elapsed_hours": hours(),
             "peak_vram_gb": float(torch.cuda.max_memory_allocated() / 1e9)}

    jp = f"{cfg.OUT_DIR}/results_{tag}.json"
    json.dump(final, open(jp, "w"), indent=2, default=str)

    flat_rows = []
    for r in cfg.R_MULTIPLIERS:
        R_ = RESULTS[f"R{r}"]
        row = {"model": "S1xS2-chain", "stage": 3, "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD,
               "set": f"R{r}", "R": r, "n_pairs": R_["n_pairs"], "n_scored": R_["n_scored"],
               "independent_draws": cfg.INDEPENDENT_DRAWS,
               "layout_mIoU": R_["layout"]["mIoU"], "layout_mF1": R_["layout"]["mF1"],
               "layout_OA": R_["layout"]["OA"], "layout_kappa": R_["layout"]["kappa"],
               "miou_ceiling": R_["layout"]["miou_ceiling"],
               "layout_ratio": R_["layout"]["layout_ratio"],
               "clDice_Road": R_["layout"]["clDice_Road"],
               "clDice_River": R_["layout"]["clDice_River"],
               "miou_scorer2": R_["layout"].get("miou_scorer2"),
               "scorer_agreement": R_["layout"].get("pixel_agreement"),
               "KID_val": R_["KID_val"]["mean"], "KID_val_std": R_["KID_val"]["std"],
               "KID_train": R_["KID_train"]["mean"],
               "FID_val": R_["FID_val"], "FID_train": R_["FID_train"],
               "mem_feat_ratio": R_["memorization"]["feat_ratio"],
               "mem_pixel_ratio": R_["memorization"]["pixel_ratio"],
               "mem_mask_ratio": R_["memorization"]["mask_ratio"],
               "mask_KL": R_["mask_KL_vs_real"], "mask_SHDI": R_["mask_SHDI"],
               "ddim_steps": DDIM_STEPS, "guidance": GUIDANCE, "cond_scale": COND_SCALE,
               "canny": USE_CANNY, "s1_ddim_steps": S1_STEPS,
               "pp_knobs": "-".join(map(str, PP_KNOBS)),
               "train_sha1": split_sha1[:12],
               "s1_repo": cfg.S1_REPO, "s2_repo": cfg.S2_REPO}
        for i, nm in enumerate(cfg.CLASS_NAMES):
            row[f"IoU_{nm}"] = R_["layout"]["per_class"][nm]["IoU"]
            row[f"maskshare_{nm}"] = R_["mask_class_share"][i]
            row[f"presence_{nm}"] = R_["mask_presence_rate"][i]
        flat_rows.append(row)

    cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
    pd.DataFrame(flat_rows).to_csv(cp, index=False)
    lp = f"{cfg.OUT_DIR}/ladder_{tag}.csv"
    SUMMARY.to_csv(lp, index=False)

    for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"),
                 (lp, f"ladder_{tag}.csv"),
                 (f"{cfg.OUT_DIR}/provenance.json", "provenance.json"),
                 (f"{cfg.OUT_DIR}/fold{cfg.FOLD}_split.json", f"fold{cfg.FOLD}_split.json")]:
        if os.path.exists(p): hf_push(p, n, "stage 3 results")
    print(f"\nresults -> {jp}")
    print(pd.DataFrame(flat_rows)[["set", "n_pairs", "layout_mIoU", "layout_ratio",
                                   "KID_val", "FID_val", "mem_feat_ratio"]].to_string(index=False))
else:
    print("ladder incomplete -- nothing persisted")

## 21. Stage-4 hand-off package

**This is the interface contract. Do not change anything in this cell without telling Stage 4.**

Each R set is tarred as `fake_pairs_R{r}_{tag}.tar.gz` with the internal layout:

```
fake_pairs_R{r}/
    genimage0.png    512x512x3 uint8, BGR on disk (cv2), exactly how the part-1 loaders read images
    genmask0.png     512x512   uint8, values {0..5} semantic + 255 ignore
    genimage1.png
    genmask1.png
    ...
```

Same values, same channel order and same dtypes as the real cached pairs, so a Stage-4 dataloader
concatenates them with `records` and needs no adapter whatsoever. `255` is `ignore_index`, which is
already what the part-1 loss expects.

**Sharding by R, not one blob.** Stage 4 will run five models across three R values across several
sessions. Pulling a 2.2 GB archive every session to use a third of it is a waste; each shard is
pulled once.

**Two homes.** The Hub is the archive and carries the provenance. For actually running Stage 4,
`Save Version` this notebook and attach `/kaggle/working/fake_pairs` as a Kaggle dataset — a
local Kaggle input mounts instantly, and a 2.2 GB download at the top of every Stage-4 session
would cost more than the training.

The card written here is a **dataset card**: what these pairs are, what produced them, what they may
and may not be used for. The "may not" matters: these pairs must never enter a validation or test
split, in any fold, ever. They are training-side augmentation only.

In [ ]:
def build_shard(r):
    src = R_DIR(r)
    tarp = f"{cfg.OUT_DIR}/fake_pairs_R{r}_{tag}.tar.gz"
    if os.path.exists(tarp): os.remove(tarp)
    with tarfile.open(tarp, "w:gz") as tf:
        tf.add(src, arcname=f"fake_pairs_R{r}")
    return tarp

if ALL_DONE:
    manifest = {
        **PROVENANCE,
        "produced": {f"R{r}": {"n_pairs": R_SIZES[r],
                               "dir": f"fake_pairs_R{r}",
                               "layout_mIoU": RESULTS[f"R{r}"]["layout"]["mIoU"],
                               "layout_ratio": RESULTS[f"R{r}"]["layout"]["layout_ratio"],
                               "KID_val": RESULTS[f"R{r}"]["KID_val"]["mean"],
                               "FID_val": RESULTS[f"R{r}"]["FID_val"],
                               "memorization_feat_ratio":
                                   RESULTS[f"R{r}"]["memorization"]["feat_ratio"]}
                     for r in cfg.R_MULTIPLIERS},
        "usage": ("TRAINING SIDE ONLY. These pairs must never enter a validation or test split, "
                  "in any fold. They were generated by models trained on block fold 1's TRAIN "
                  "tiles; mixing them into any evaluation set is leakage."),
        "naming": "genimage{i}.png + genmask{i}.png, i in 0..n-1",
    }
    mp = f"{cfg.OUT_DIR}/fake_pairs_manifest.json"
    json.dump(manifest, open(mp, "w"), indent=2, default=str)
    hf_push(mp, "fake_pairs_manifest.json", "stage 4 hand-off manifest")

    if cfg.PUSH_SHARDS:
        for r in cfg.R_MULTIPLIERS:
            t0 = time.time(); tarp = build_shard(r)
            print(f"R{r}: {os.path.getsize(tarp)/1e6:.0f} MB in {time.time()-t0:.0f}s")
            ok = hf_push(tarp, os.path.basename(tarp), f"fake pairs R{r}")
            if ok:
                os.remove(tarp)          # free the working disk before the next shard
                print(f"  local tarball removed (it lives on {HF_REPO} now)")
    else:
        print("PUSH_SHARDS off -- tarballs not built. Attach /kaggle/working/fake_pairs as a "
              "Kaggle dataset instead.")

    # ---- dataset card ----
    L = ["---", "tags: [remote-sensing, synthetic-data, segmentation, diffusion, geonusaf]",
         "license: cc-by-4.0", "---", "",
         f"# GeoNUSAF Stage 3 - synthetic image-mask pairs ({cfg.SPLIT_MODE} fold {cfg.FOLD})", "",
         "Synthetic Kathmandu Valley satellite image / land-use mask pairs, produced by chaining",
         "the Part-2 Stage-1 mask diffusion model into the Stage-2 mask-conditioned image",
         "generator. Intended as training-side augmentation for the Part-1 segmentation models.",
         "", "| field | value |", "|---|---|",
         f"| mask generator | `{cfg.S1_REPO}` (epoch {S1_CK.get('epoch')}) |",
         f"| image generator | `{cfg.S2_REPO}` (step {S2_CK.get('step')}) |",
         f"| split | {cfg.SPLIT_MODE} fold {cfg.FOLD}, train_sha1 `{split_sha1[:12]}` |",
         f"| real training pairs | {N_REAL} |",
         f"| sets | " + ", ".join(f"R{r} = {R_SIZES[r]} pairs" for r in cfg.R_MULTIPLIERS) + " |",
         f"| draws | {'independent per set' if cfg.INDEPENDENT_DRAWS else 'nested prefixes'} |",
         f"| mask sampling | DDIM {S1_STEPS}, pool ratio {cfg.MASK_POOL_RATIO}, "
         f"post-processing modal{PP_KNOBS[0]}/close{PP_KNOBS[1]}/thin{PP_KNOBS[2]}/bulk{PP_KNOBS[3]} |",
         f"| image sampling | DDIM {DDIM_STEPS}, guidance {GUIDANCE}, cond scale {COND_SCALE}, "
         f"canny {USE_CANNY} @ {CANNY_SCALE} |",
         f"| resolution | {cfg.IMG_SIZE}x{cfg.IMG_SIZE}, 1.374 m/px |",
         f"| scorer | `{cfg.SEG_REPO_BLOCK}`, ceiling mIoU {CEIL['mIoU']:.4f} |", ""]
    L += ["## Quality", "", "| set | layout mIoU | ratio to ceiling | KID vs val | FID vs val | "
          "memorization ratio (feat / pixel / mask) |", "|---|---|---|---|---|---|"]
    for r in cfg.R_MULTIPLIERS:
        R_ = RESULTS[f"R{r}"]; M = R_["memorization"]
        L.append(f"| R{r} | {R_['layout']['mIoU']:.4f} | {R_['layout']['layout_ratio']:.3f} | "
                 f"{R_['KID_val']['mean']:.5f} +/- {R_['KID_val']['std']:.5f} | "
                 f"{R_['FID_val']:.1f} | {M['feat_ratio']:.2f} / {M['pixel_ratio']:.2f} / "
                 f"{M['mask_ratio']:.2f} |")
    L += ["", "Memorization ratio = mean nearest-neighbour distance of a fake to the training set,",
          "divided by the same statistic for a **held-out real** tile. At or above 1.0 the fakes",
          "are no closer to the training data than a genuinely new real tile is.", "",
          "## Contract", "",
          f"- `genimage{{i}}.png` - {cfg.IMG_SIZE}x{cfg.IMG_SIZE}x3 `uint8`, **BGR on disk** "
          "(written with `cv2.imwrite`, read with `cv2.IMREAD_COLOR`)",
          f"- `genmask{{i}}.png` - {cfg.IMG_SIZE}x{cfg.IMG_SIZE} single-channel `uint8`, "
          "values `{0..5}` semantic + `255` ignore",
          "- class order: " + ", ".join(f"{i}={n}" for i, n in enumerate(cfg.CLASS_NAMES)),
          "", "## Usage restriction", "",
          "**Training side only.** These pairs must never enter a validation or test split in any",
          "fold. The generators saw block fold 1's TRAIN tiles; putting their output into an",
          "evaluation set is leakage."]
    cardp = f"{cfg.OUT_DIR}/README.md"; open(cardp, "w").write("\n".join(L))
    hf_push(cardp, "README.md", "dataset card")

    print(f"\nStage 4 input: pull the shards from {HF_REPO}, or Save Version and attach")
    print(f"{cfg.PAIR_DIR} as a Kaggle dataset (faster, and what you want for a multi-session run).")
else:
    print("ladder incomplete -- hand-off package not built")

## 22. Standalone chained sampler from Hugging Face

Self-contained, exactly like the inference cells in the part-1 notebooks and in both upstream
stages: point it at the two repos, ask for N pairs, get PNGs back. Nothing above this cell is
required — both architectures are rebuilt from the Hub plus the checkpoints' own embedded configs,
and the sampling hyper-parameters are inherited the same way section 12 inherits them.

Paste it into a fresh notebook as-is when Stage 4 wants a fourth ratio, a variance estimate at a
different seed, or a handful of extra examples for a figure.

Note what it does **not** do: no post-processing sweep, no distribution-matched selection, no
evaluation. It is the raw chain. Masks from it will have the unfiltered class marginal, which is
the right thing for a variance estimate and the wrong thing for a Stage-4 training set — for that,
use the shards from section 21.

In [ ]:
def chain_from_hf(s1_repo, s2_repo, n, out_dir, token=None, seed=0, device="cuda",
                  s1_steps=None, steps=None, guidance=None, batch=4, pp=(0, 3, 30, 0)):
    import torch, numpy as np, cv2, os, math
    from pathlib import Path
    from diffusers import (UNet2DModel, UNet2DConditionModel, ControlNetModel,
                           AutoencoderKL, DDIMScheduler)
    from transformers import CLIPTextModel, CLIPTokenizer
    from huggingface_hub import hf_hub_download
    from peft import LoraConfig
    from scipy import ndimage as ndi
    os.makedirs(out_dir, exist_ok=True)

    # ---- stage 1 ----
    ck1 = torch.load(hf_hub_download(s1_repo, "best.pt", token=token), map_location=device)
    c1 = ck1["cfg"]
    m1 = UNet2DModel.from_config(ck1["model_config"]).to(device).eval()
    m1.load_state_dict(ck1["model_state"]); m1.requires_grad_(False)
    s1_steps = s1_steps or c1.get("FINAL_DDIM_STEPS", 200)
    kw = dict(num_train_timesteps=c1["TIMESTEPS"], beta_start=c1["BETA_START"],
              beta_end=c1["BETA_END"], beta_schedule="linear",
              prediction_type=c1["PREDICTION_TYPE"], clip_sample=True, steps_offset=0)
    if c1.get("ZERO_TERMINAL_SNR", False):
        kw.update(rescale_betas_zero_snr=True, timestep_spacing="trailing", set_alpha_to_one=True)
    else:
        kw.update(set_alpha_to_one=False)
    masks = []
    with torch.no_grad():
        for s in range(0, n, 32):
            b = min(32, n - s)
            gg = torch.Generator(); gg.manual_seed(seed + s)
            x = torch.randn(b, c1["N_CH"], c1["GEN_SIZE"], c1["GEN_SIZE"],
                            generator=gg).to(device)
            sc = DDIMScheduler(**kw); sc.set_timesteps(s1_steps)
            for t in sc.timesteps:
                with torch.cuda.amp.autocast():
                    p = m1(x, t).sample
                x = sc.step(p.float(), t, x, eta=0.0).prev_sample
            masks.append(x.clamp(-1, 1).argmax(1).to(torch.uint8).cpu().numpy())
    masks = np.concatenate(masks)
    del m1; torch.cuda.empty_cache()

    # light despeckle only: bridge thin-class gaps and drop specks
    def _pp(a, close_k, min_px):
        out = a.copy()
        for c in (1, 2):
            m = a == c
            if not m.any(): continue
            if close_k > 1: m = ndi.binary_closing(m, structure=np.ones((close_k, close_k), bool))
            lab, _ = ndi.label(m, structure=np.ones((3, 3), int))
            ar = np.bincount(lab.ravel()); ar[0] = 0
            m = np.isin(lab, np.where(ar >= min_px)[0])
            out = np.where(m, c, out).astype(np.uint8)
        return out
    masks = np.stack([_pp(a, pp[1], pp[2]) for a in masks])
    big = np.stack([cv2.resize(a, (c1["OUT_SIZE"], c1["OUT_SIZE"]),
                               interpolation=cv2.INTER_NEAREST) for a in masks])
    big = np.where(big == c1["IGNORE_CODE"], c1["IGNORE_INDEX"], big).astype(np.uint8)

    # ---- stage 2 ----
    ck2 = torch.load(hf_hub_download(s2_repo, "best.pt", token=token), map_location="cpu")
    c2 = ck2["cfg"]; cond_ch = 7
    steps    = steps    or c2.get("DDIM_STEPS", 30)
    guidance = guidance if guidance is not None else c2.get("GUIDANCE", 7.5)
    u = UNet2DConditionModel.from_pretrained(c2["SD_MODEL_ID"], subfolder="unet",
                                             torch_dtype=torch.float16).to(device)
    u.requires_grad_(False)
    cn = ControlNetModel.from_pretrained(c2["CONTROLNET_ID"], torch_dtype=torch.float32).to(device)
    o = cn.controlnet_cond_embedding.conv_in
    cn.controlnet_cond_embedding.conv_in = torch.nn.Conv2d(
        cond_ch, o.out_channels, kernel_size=o.kernel_size, stride=o.stride,
        padding=o.padding).to(device)
    cn.register_to_config(conditioning_channels=cond_ch)
    if ck2.get("lora_state"):
        u.add_adapter(LoraConfig(r=c2["LORA_RANK"], lora_alpha=c2["LORA_ALPHA"], lora_dropout=0.0,
                                 init_lora_weights="gaussian",
                                 target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
        for p in u.parameters():
            if p.requires_grad: p.data = p.data.float()
        u.load_state_dict({k: v.float() for k, v in ck2["lora_state"].items()}, strict=False)
    cn.load_state_dict({k: v.float() for k, v in ck2["control_state"].items()})
    u.eval(); cn.eval()
    v  = AutoencoderKL.from_pretrained(c2["SD_MODEL_ID"], subfolder="vae").to(device).eval()
    te = CLIPTextModel.from_pretrained(c2["SD_MODEL_ID"], subfolder="text_encoder",
                                       torch_dtype=torch.float16).to(device).eval()
    tk = CLIPTokenizer.from_pretrained(c2["SD_MODEL_ID"], subfolder="tokenizer")
    scfg = DDIMScheduler.from_pretrained(c2["SD_MODEL_ID"], subfolder="scheduler").config
    eye = np.eye(cond_ch, dtype=np.float32)
    NOUN_ = {0: "residential rooftops", 1: "road network", 2: "river", 3: "forest",
             4: "bare unused land", 5: "agricultural fields"}

    def cap(lab):
        code = np.where(lab == 255, 6, lab).astype(np.uint8)
        h = np.bincount(code.ravel(), minlength=7)[:6].astype(np.float64); s = h / max(h.sum(), 1)
        keep = [int(c) for c in np.argsort(-s) if s[c] >= 0.02][:3]
        for c in (1, 2):
            if s[c] >= 0.005 and c not in keep: keep.append(c)
        parts = []
        for c in sorted(keep, key=lambda c: -s[c]):
            q = ("dense " if s[c] >= 0.35 else ("scattered " if s[c] < 0.10 else "")) \
                if c in (0, 3, 4, 5) else ""
            parts.append(q + NOUN_[c])
        return "aerial satellite view of Kathmandu Valley, " + ", ".join(parts)

    def embed(caps):
        t = tk(caps, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
        with torch.no_grad(): return te(t.input_ids.to(device))[0]

    written = 0
    with torch.no_grad():
        for s in range(0, n, batch):
            chunk = big[s:s + batch]; b = len(chunk)
            code = np.where(chunk == 255, 6, chunk).astype(np.uint8)
            cond = torch.from_numpy(eye[code].transpose(0, 3, 1, 2)).to(device)
            emb, neg = embed([cap(m) for m in chunk]), embed([""] * b)
            sch = DDIMScheduler.from_config(scfg); sch.set_timesteps(steps, device=device)
            gg = torch.Generator(); gg.manual_seed(seed + 10007 + s)
            lat = torch.randn(b, 4, 64, 64, generator=gg).to(device) * sch.init_noise_sigma
            for t in sch.timesteps:
                x = sch.scale_model_input(torch.cat([lat] * 2), t)
                with torch.cuda.amp.autocast():
                    d, mm = cn(x, t, encoder_hidden_states=torch.cat([neg, emb]),
                               controlnet_cond=torch.cat([cond, cond]),
                               conditioning_scale=c2.get("COND_SCALE", 1.0), return_dict=False)
                    e = u(x, t, encoder_hidden_states=torch.cat([neg, emb]),
                          down_block_additional_residuals=[a.to(u.dtype) for a in d],
                          mid_block_additional_residual=mm.to(u.dtype)).sample
                e = e.float(); e_u, e_c = e.chunk(2); e = e_u + guidance * (e_c - e_u)
                lat = sch.step(e, t, lat).prev_sample
            img = v.decode((lat / v.config.scaling_factor).float()).sample
            img = ((img.float() / 2 + 0.5).clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1) * 255)
            img = img.round().astype(np.uint8)
            for k in range(b):
                cv2.imwrite(f"{out_dir}/genimage{s+k}.png",
                            cv2.cvtColor(img[k], cv2.COLOR_RGB2BGR),
                            [cv2.IMWRITE_PNG_COMPRESSION, 3])
                cv2.imwrite(f"{out_dir}/genmask{s+k}.png", chunk[k],
                            [cv2.IMWRITE_PNG_COMPRESSION, 3])
                written += 1
    return written

# --- example: 8 fresh pairs at a seed nothing above used ---
_demo_dir = "/kaggle/working/standalone_demo"
_n = chain_from_hf(cfg.S1_REPO, cfg.S2_REPO, 8, _demo_dir, token=HF_TOKEN, seed=123456)
print(f"standalone chain wrote {_n} pairs to {_demo_dir}")
_di = [cv2.cvtColor(cv2.imread(f"{_demo_dir}/genimage{i}.png", cv2.IMREAD_COLOR),
                    cv2.COLOR_BGR2RGB) for i in range(4)]
_dm = [cv2.imread(f"{_demo_dir}/genmask{i}.png", cv2.IMREAD_UNCHANGED) for i in range(4)]
fig, ax = plt.subplots(2, 4, figsize=(11, 5.6))
for i in range(4):
    ax[0, i].imshow(colorize(_dm[i])); ax[0, i].axis("off")
    ax[1, i].imshow(_di[i]); ax[1, i].axis("off")
ax[0, 0].set_ylabel("mask"); ax[1, 0].set_ylabel("image")
fig.suptitle("standalone chained sampler - fake mask (top) -> fake image (bottom)", fontsize=11)
fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
plt.tight_layout(rect=[0, 0.06, 1, 0.95]); plt.show()

## 23. Qualitative grids — and the memorization figure

Two figures, in the order a reader should meet them.

**Fake pairs.** Generated mask on top, generated image below. Same palette as every part-1 figure,
ignore renders black, so these sit next to those without a second legend. What to look for, in
order:

1. Do the layouts read as Kathmandu? Residential should dominate, roads should be a connected
   network rather than scattered dashes, rivers should be continuous channels rather than ponds.
2. Does the image *obey* the mask? A generated river that is painted as rooftops is a control-branch
   failure, and it is exactly what the layout mIoU number is measuring.
3. Does the texture read as satellite imagery at 1.37 m/px, or as a generic aerial photo?

**The memorization figure is the one that matters.** For the `NN_TOPK_FIG` fakes with the *smallest*
distance to any training image — the worst cases, not average ones — it shows the fake beside its
nearest real training tile. This is the figure a reviewer will look for and the one you cannot argue
your way past with a summary statistic. If the pairs look like the same place, the ratio in section
19 was not telling you what you hoped. Put this figure in the paper regardless of what it shows.

In [ ]:
if ALL_DONE:
    r0 = cfg.R_MULTIPLIERS[0]
    idx0 = list(range(8))
    G_I = [cv2.cvtColor(cv2.imread(_pair_paths(r0, i)[0], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
           for i in idx0]
    G_M = [cv2.imread(_pair_paths(r0, i)[1], cv2.IMREAD_UNCHANGED) for i in idx0]

    fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
    for i in range(6):
        ax[0, i].imshow(colorize(G_M[i])); ax[0, i].axis("off")
        ax[1, i].imshow(G_I[i]); ax[1, i].axis("off")
        ax[0, i].set_title(f"pair {i}", fontsize=9)
    fig.suptitle(f"{RUN_NAME} / R{r0} - fake mask (top) -> fake image (bottom)", fontsize=11)
    fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    pq = f"{cfg.OUT_DIR}/qual_pairs_{tag}.png"
    plt.savefig(pq, dpi=130, bbox_inches="tight"); plt.show()

    # ---- real triptych for side-by-side comparison ----
    fig = triptych(list(REAL_VAL_RGB[:6]), list(REAL_VAL_LAB[:6]),
                   G_I[:6], n=6,
                   title=f"real val tile / its mask / an unrelated fake ({RUN_NAME})")
    pr_ = f"{cfg.OUT_DIR}/qual_real_vs_fake_{tag}.png"
    plt.savefig(pr_, dpi=130, bbox_inches="tight"); plt.show()

    # ---- the memorization figure: WORST cases, not average ones ----
    k = min(cfg.NN_CHECK_N, R_SIZES[r0])
    G_RGB_K = np.stack([cv2.cvtColor(cv2.imread(_pair_paths(r0, i)[0], cv2.IMREAD_COLOR),
                                     cv2.COLOR_BGR2RGB) for i in range(k)])
    FGk = inception_feats(G_RGB_K)
    dk, jk = nn_feat(FGk, F_TRAIN)
    worst = np.argsort(dk)[:cfg.NN_TOPK_FIG]
    nshow = min(cfg.NN_TOPK_FIG, len(worst))
    fig, ax = plt.subplots(2, nshow, figsize=(2.1 * nshow, 4.6))
    ax = np.atleast_2d(ax)
    for c, w in enumerate(worst[:nshow]):
        ax[0, c].imshow(G_RGB_K[w]); ax[0, c].axis("off")
        ax[0, c].set_title(f"fake {w}\nd={dk[w]:.3f}", fontsize=8)
        ax[1, c].imshow(REAL_TRAIN_RGB[jk[w]])       # rows are in tr_recs order
        ax[1, c].axis("off")
        ax[1, c].set_title(f"nearest real\n{tr_recs[jk[w]][2]}", fontsize=8)
    fig.suptitle(f"MEMORIZATION CHECK - the {nshow} fakes CLOSEST to the training set "
                 f"(baseline for held-out real: {BASE['feat']:.3f})", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    pm = f"{cfg.OUT_DIR}/memorization_worst_{tag}.png"
    plt.savefig(pm, dpi=130, bbox_inches="tight"); plt.show()

    for p, n in [(pq, f"qual_pairs_{tag}.png"), (pr_, f"qual_real_vs_fake_{tag}.png"),
                 (pm, f"memorization_worst_{tag}.png")]:
        hf_push(p, n, "qualitative figures")
    if WANDB:
        wandb.log({"qual/pairs": wandb.Image(pq), "qual/real_vs_fake": wandb.Image(pr_),
                   "qual/memorization_worst": wandb.Image(pm)})
    print("\nLook at the bottom figure before you believe anything above it.")
else:
    print("ladder incomplete -- figures skipped")

## 24. Distribution, ladder and memorization figures

Four panels, in the order a reader should meet them.

1. **Class proportions**, real vs each generated set, on a log axis. Without the log axis River at
   0.47% is invisible next to Residential at 52%, which is precisely the class the figure exists to
   show.
2. **Per-mask presence rate** for the two thin classes. This is the panel that answers "can Stage 4
   possibly gain on Road and River", and it is separate from panel 1 on purpose: a set can match the
   pooled share while almost every individual mask is riverless.
3. **The R ladder**: layout mIoU, KID and memorization ratio against R. These should be flat. Three
   independent draws from one generator have no reason to differ, and a trend here would mean the
   pools differ for a reason you have not identified — fix that before Stage 4, not after.
4. **Memorization distances**: the distribution of nearest-neighbour distance for fakes against the
   same distribution for held-out real tiles, at all three levels. Overlapping histograms are the
   result you want. A fake distribution shifted left of the real one is the failure case, and the
   left *tail* matters more than the mean.

In [ ]:
if ALL_DONE:
    # ---- panel 1: class proportions, log axis ----
    x = np.arange(cfg.N_CODES); w = 0.8 / (len(cfg.R_MULTIPLIERS) + 1)
    fig, ax = plt.subplots(figsize=(9.5, 3.4))
    ax.bar(x - 0.4 + w / 2, 100 * P_REAL, w, label="real (fold-1 train)", color="k", alpha=0.75)
    for j, r in enumerate(cfg.R_MULTIPLIERS):
        s = np.array(RESULTS[f"R{r}"]["mask_class_share"])
        ax.bar(x - 0.4 + w * (j + 1.5), 100 * s, w, label=f"R{r}")
    ax.set_xticks(x); ax.set_xticklabels(CODE_NAMES, rotation=20, ha="right")
    ax.set_yscale("log"); ax.set_ylabel("pixel share (%, log)")
    ax.set_title("class proportions: real vs the three generated sets")
    ax.legend(fontsize=8, ncol=4); plt.tight_layout()
    p1 = f"{cfg.OUT_DIR}/fig_class_shares_{tag}.png"
    plt.savefig(p1, dpi=130, bbox_inches="tight"); plt.show()

    # ---- panel 2: per-mask presence rate of the thin classes ----
    pr_real = presence_rate(TRAIN_128)
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    lbl = ["Road", "River"]; xs = np.arange(2)
    ax.bar(xs - 0.4 + w / 2, [100 * pr_real[1], 100 * pr_real[2]], w,
           label="real", color="k", alpha=0.75)
    for j, r in enumerate(cfg.R_MULTIPLIERS):
        pr = np.array(RESULTS[f"R{r}"]["mask_presence_rate"])
        ax.bar(xs - 0.4 + w * (j + 1.5), [100 * pr[1], 100 * pr[2]], w, label=f"R{r}")
    ax.set_xticks(xs); ax.set_xticklabels(lbl); ax.set_ylabel("% of masks containing the class")
    ax.set_title("per-mask presence rate of the thin classes")
    ax.legend(fontsize=8, ncol=4); plt.tight_layout()
    p2 = f"{cfg.OUT_DIR}/fig_presence_{tag}.png"
    plt.savefig(p2, dpi=130, bbox_inches="tight"); plt.show()

    # ---- panel 3: the ladder ----
    Rs = list(cfg.R_MULTIPLIERS)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    axes[0].plot(Rs, [RESULTS[f"R{r}"]["layout"]["mIoU"] for r in Rs], "o-")
    axes[0].axhline(CEIL["mIoU"], ls="--", c="k", lw=1, label="real ceiling")
    axes[0].set_ylabel("layout mIoU"); axes[0].legend(fontsize=8)
    axes[1].errorbar(Rs, [RESULTS[f"R{r}"]["KID_val"]["mean"] for r in Rs],
                     yerr=[RESULTS[f"R{r}"]["KID_val"]["std"] for r in Rs], fmt="o-")
    axes[1].set_ylabel("KID vs held-out real")
    for lv, mk in (("feat", "o-"), ("pixel", "s-"), ("mask", "^-")):
        axes[2].plot(Rs, [RESULTS[f"R{r}"]["memorization"][f"{lv}_ratio"] for r in Rs], mk, label=lv)
    axes[2].axhline(1.0, ls="--", c="k", lw=1)
    axes[2].set_ylabel("memorization ratio"); axes[2].legend(fontsize=8)
    for a in axes: a.set_xlabel("R (x real training set)"); a.set_xticks(Rs)
    fig.suptitle("the R ladder - these should be FLAT (three independent draws, one generator)",
                 fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    p3 = f"{cfg.OUT_DIR}/fig_ladder_{tag}.png"
    plt.savefig(p3, dpi=130, bbox_inches="tight"); plt.show()

    # ---- panel 4: memorization distance distributions ----
    r0 = cfg.R_MULTIPLIERS[0]
    k = min(cfg.NN_CHECK_N, R_SIZES[r0])
    G_RGB_K = np.stack([cv2.cvtColor(cv2.imread(_pair_paths(r0, i)[0], cv2.IMREAD_COLOR),
                                     cv2.COLOR_BGR2RGB) for i in range(k)])
    dists = {
        "Inception feature": (nn_feat(inception_feats(G_RGB_K), F_TRAIN)[0],
                              d_feat_base),
        "raw pixel (64px)":  (nn_pixel(thumbs(G_RGB_K), T_TRAIN)[0], d_pix_base),
        "mask (32px)":       (nn_mask(MASKS128[r0][:k], TRAIN_128)[0], d_mask_base),
    }
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.0))
    for a, (nm, (dg, db)) in zip(axes, dists.items()):
        lo = min(dg.min(), db.min()); hi = max(dg.max(), db.max())
        bins = np.linspace(lo, hi, 40)
        a.hist(db, bins=bins, alpha=0.55, label="held-out real", density=True, color="k")
        a.hist(dg, bins=bins, alpha=0.55, label="generated", density=True)
        a.set_title(nm, fontsize=10); a.set_xlabel("min distance to train set")
        a.legend(fontsize=8)
    fig.suptitle("memorization - overlapping distributions are the result you want; "
                 "watch the LEFT tail", fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    p4 = f"{cfg.OUT_DIR}/fig_memorization_{tag}.png"
    plt.savefig(p4, dpi=130, bbox_inches="tight"); plt.show()

    for p, n in [(p1, f"fig_class_shares_{tag}.png"), (p2, f"fig_presence_{tag}.png"),
                 (p3, f"fig_ladder_{tag}.png"), (p4, f"fig_memorization_{tag}.png")]:
        hf_push(p, n, "stage 3 figures")
    if WANDB:
        wandb.log({"fig/class_shares": wandb.Image(p1), "fig/presence": wandb.Image(p2),
                   "fig/ladder": wandb.Image(p3), "fig/memorization": wandb.Image(p4)})
        wandb.finish()
else:
    print("ladder incomplete -- figures skipped")

## 25. Run checklist

1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; `sugam24/satellite-datasets`
   attached; GPU T4 selected. `MULTI_GPU` stays `False` unless you are deliberately experimenting.
0. **Check the mask pools first.** Section 11 should print `mask pool restored from ...` for each
   R, not `=== R3: sampling 6030 masks ===`. If it starts sampling, stop the run: the pools are on
   the Hub and something is not finding them, and you are about to spend eight hours re-deriving
   them for nothing.
2. **Wait for Stage 2 to finish.** This notebook reads whatever `best.pt` is in the Stage-2 repo at
   the moment it runs. If you start it while Stage 2 is still training, you will generate 4824 pairs
   from a checkpoint you are about to replace, and there is no way to tell afterwards which one
   produced them except the `step` recorded in the manifest. Section 12 prints that step — check it
   against the Stage-2 run before you let section 18 start.
3. Section 5 must print **channel-order check PASSED** and an unmapped share of `0.000000%`.
4. Section 8 must print a `train_sha1` that matches what Stage 1 and Stage 2 printed. Paste it into
   `cfg.UPSTREAM_TRAIN_SHA1` to turn that into an assert rather than a thing you eyeballed once.
5. Section 11 prints `Road_in` and `River_in` — the **per-mask presence rates**. Read these before
   anything else. If River appears in only a few percent of masks, Stage 4 cannot gain on River no
   matter how good the images look, and it is better to know that now than after five model
   retrainings. That is a Stage-1 problem, not something this notebook can fix.
6. Section 12 prints the resolved sampling hyper-parameters and whether each was inherited or
   overridden. These go in the paper verbatim.
7. Section 14 prints the projected wall-clock and the session count. **Read it before walking
   away.** If it says three sessions, plan for three.
8. Section 18 stops cleanly at `MAX_HOURS`. When it does: Save Version, start a new session, rerun
   top to bottom. Sections 6 to 12 are all cache-or-download; section 17 resumes at the exact pair.
9. Section 19 only runs when the whole ladder is on disk. Read the three memorization ratios first,
   then the layout ratio, then KID.
10. Section 21 pushes one tarball per R set and deletes each local copy after a successful push, so
    the working disk does not fill. Also **Save Version and attach `/kaggle/working/fake_pairs` as a
    Kaggle dataset** — that is what Stage 4 should actually read from.

### Knobs, and when to touch them

| symptom | knob |
|---|---|
| a session died and you want to know what survived | nothing to do: every pair up to the last checkpoint is on the Hub, and section 17 pulls them back automatically |
| pushes are failing | lower `CHECKPOINT_EVERY` so less is at risk per push, and check the HF token has write access. The run continues either way |
| you want to try both GPUs | `MULTI_GPU = True`. Section 14 benches each device and drops any that fails, so the worst case is a wasted bench and a single-GPU run — never a lost session |
| OOM during sampling | `IMG_BATCH` 8 -> 4 -> 2 |
| session too short for one whole set | nothing; the resume handles it |
| River presence rate near zero | Stage 1 problem. `MASK_POOL_RATIO` up to 4.0 gives `greedy_match` more to select from, but it cannot conjure rivers that were never sampled |
| layout ratio below the floor | check the condition-source ablation in section 19 first. If real masks score much better, the Stage-1 masks are out of distribution for the control branch and the fix is upstream |
| want a fourth ratio or a variance estimate | section 22's standalone chain, at a fresh seed |
| disk filling | `PUSH_SHARDS` on (it deletes each tarball after pushing), or drop R3 |

### What goes in the paper from this notebook

- The provenance block: both checkpoints, both steps, the resolved sampling settings, the split
  SHA1. Stage 3 has no hyper-parameters of its own worth reporting except these.
- The R-ladder table from section 19 and the ladder figure from section 24.
- **The memorization figure and all three ratios, whatever they say.** A synthetic-augmentation
  paper without a memorization check is not a finished paper, and a reviewer will ask.
- The condition-source ablation, as a one-line result: how much layout fidelity is lost by feeding
  the image generator a *generated* mask rather than a real one. That number is the cost of chaining,
  and nobody else has reported it for this kind of pipeline.

### What this notebook deliberately leaves to Stage 4

Mixing ratios inside the training loader, class-balanced sampling over the combined real+fake set,
and the comparison against CutMix / Copy-Paste / Cutout. None of that belongs here. Stage 3's only
job is to produce pairs that are honestly labelled, honestly measured, and traceable.